# Notebook 2: Huấn luyện Unpaired StegoGAN (`RGB 3-ch → IR 1-ch`, CVPR 2024) & Đánh giá so sánh với CycleGAN (Nạp JSON từ Notebook 1)

Notebook độc lập thứ 2, tự chứa toàn bộ mã nguồn (**thuần PyTorch / Torchvision**, upload thẳng lên **Kaggle / Colab / Server Local** không cần `git clone`) triển khai **StegoGAN** (*Leveraging Steganography for Non-Bijective Image-to-Image Translation*, Wu et al., CVPR 2024 — `sian-wusidi/StegoGAN`):

### Điểm mạnh riêng biệt của Hướng 2 (StegoGAN) & Cách nối tiếp từ Notebook 1:
1. **Xử lý trực diện tính Phi song ánh (*Non-Bijective*) giữa RGB (3 kênh) và Thermal IR (1 kênh):**
   - Trong CycleGAN truyền thống, các đặc trưng chỉ có ở miền nhiệt (điểm nóng động cơ, bức xạ người trong bóng tối) hoặc chỉ có ở miền RGB (màu sơn, bóng nắng, chữ viết) buộc mạng sinh phải **giấu tin tần số cao** hoặc **bịa đặt vật thể giả (*hallucination*)**.
   - **StegoGAN** học thêm mô-đun **`NetMatchability` (`latent_mask`)** ở nhánh ngược để tự động phát hiện và tách bỏ phần thông tin không đối ứng, nhờ đó nhánh sinh chính **`fake_B_clean = netG_A(RGB, extrafeature=None)`** tạo ra ảnh IR **1 kênh** sạch, không nhúng nhiễu ẩn và không bịa đốm sáng giả trên nền cây/cột đèn.
2. **Đặc quyền kỹ thuật — Xuất thẳng trọng số tương thích 100% `CycleGANGenerator` của dự án:**
   - Khi suy luận sạch (`extrafeature=None`), đường đi của `ResnetMaskV1Generator` (`conv1 -> conv2/conv3 -> decoder`, 9 khối ResNet, `InstanceNorm2d`, `3ch -> 1ch`) **đồng nhất toán học 100%** với `models/cyclegan_generator.py`!
   - Notebook tự động xuất `best_net_G_A.pth` & `latest_net_G_A.pth` sang định dạng khóa phẳng của `CycleGANGenerator` (có `assert max_abs_diff < 1e-6` kiểm tra sai số số học bằng 0!), giúp bạn **cắm thẳng vào `--gan-weights` của `DAOD-Duccuong` mà không phải sửa 1 dòng code nào trong repo chính**.
3. **Tái sử dụng trực tiếp kết quả đánh giá CycleGAN từ Notebook 1:**
   - Chỉ cần gắn file **`cyclegan_baseline_eval.json`** (đã xuất ở Notebook 1) vào `CONFIG["cyclegan_eval_json"]` — Notebook 2 sẽ nạp ngay kết quả của CycleGAN (và cả mốc trần `Real IR`) mà không tốn thời gian chấm lại CycleGAN!
   - *(Tùy chọn mở rộng)* Nếu bạn gắn thêm file `comparison_flsesim_vs_cyclegan.json` từ Notebook 1 vào `CONFIG["flsesim_eval_json_optional"]`, Cell 6 sẽ tự động in **Bảng Đánh giá 3 Bên: `CycleGAN` vs `F-LSeSim` vs `StegoGAN`**!

In [ ]:
# ==============================================================================
# CELL 1: CẤU HÌNH TẬP TRUNG (CHỈ CẦN CHỈNH ĐƯỜNG DẪN TẠI ĐÂY)
# ==============================================================================
CONFIG = {
    # --- 1. Đường dẫn dữ liệu FLIR aligned & các File đầu vào trên Kaggle ---
    "data_root": "/kaggle/input/datasets/tungduong123qwe/flir-dataset/align",

    # [BẮT BUỘC] Đường dẫn checkpoint FCOS Detector TỐT NHẤT (cùng đúng file đã dùng ở Notebook 1 để so sánh công bằng!):
    "best_fcos_checkpoint": "/kaggle/input/datasets/phamduccuong05/fcos-weight/best_model.pth",

    # [NỐI TIẾP TỪ NOTEBOOK 1] Đường dẫn file cyclegan_baseline_eval.json bạn đã lưu từ Notebook 1 (để None nếu đo trực tiếp qua cyclegan_weights_fallback):
    "cyclegan_eval_json": None,

    # [TÙY CHỌN] Nếu muốn in bảng so sánh 3 bên (CycleGAN vs F-LSeSim vs StegoGAN), điền thêm file comparison_flsesim_vs_cyclegan.json từ Notebook 1:
    "flsesim_eval_json_optional": None,

    # [DỰ PHÒNG] Đo trực tiếp từ file .pth của CycleGAN hiện tại (nếu chưa có cyclegan_eval_json từ Notebook 1):
    "cyclegan_weights_fallback": "/kaggle/input/datasets/bdv212/generator-rgb-ir/latest_net_G_A.pth",

    # Kiểm tra nghiêm ngặt mọi đường dẫn & GPU ngay ở Cell 2 (tránh tốn 1.5h train rồi cuối giờ mới báo sai đường dẫn!):
    "strict_preflight_check": True,

    # Thư mục lưu trọng số StegoGAN, biểu đồ và báo cáo JSON:
    "output_dir": "/kaggle/working/stegogan_rgb2ir",

    # --- 2. Quy thức tách Disjoint Unpaired (Giống hệt 100% Notebook 1 & data/dataloaders.py) ---
    "source_fraction": 0.5,       # 50% đầu làm RGB nguồn (3ch), 50% sau làm IR đích (1ch)
    "split_seed": -1,             # -1 = giữ đúng thứ tự chuẩn data/dataloaders.py & GAN_BRIDGE_WORKFLOW.md

    # --- 3. Cấu hình huấn luyện StegoGAN (RGB 3ch -> IR 1ch, Non-Bijective Masking) ---
    # Ước tính thời gian trên Kaggle GPU T4/P100: ~52s/epoch -> 80 epochs mất ~1 giờ 15 phút (~4.1 GB VRAM).
    "run_training": True,         # Đặt False nếu chỉ muốn nạp checkpoint StegoGAN có sẵn để chạy Eval
    "eval_stegogan_weights": None,  # Nếu run_training=False, điền đường dẫn best_net_G_A.pth của StegoGAN vào đây
    "resume_train_state": None,   # Đường dẫn stegogan_resume_latest.pth nếu muốn train tiếp từ lần chạy trước

    "n_epochs": 50,               # Số epoch giữ nguyên learning rate
    "n_epochs_decay": 50,         # Số epoch giảm tuyến tính learning rate về 0 (tổng = 100 epochs ~ 103k steps)
    "batch_size": 2,              # Batch size khi train StegoGAN (~4.1 GB VRAM ở crop 288x288)
    "load_short_edge": 320,       # Resize cạnh ngắn về 320 (giữ tỷ lệ gốc 640x512 -> 400x320)
    "crop_size": 288,             # Random crop 288x288 độc lập giữa miền RGB và miền IR
    "lrG": 2e-4,                  # Chuẩn train_options.py gốc của StegoGAN (2e-4, Adam beta1=0.5, beta2=0.999)
    "lrD": 2e-4,
    "lambda_A": 10.0,             # Trọng số chu trình thuận có mặt nạ (A -> B -> A)
    "lambda_B": 10.0,             # Trọng số chu trình nghịch có mặt nạ (B -> A -> B)
    "lambda_reg": 0.25,           # Trọng số chính quy hóa thưa mặt nạ phi song ánh (chuẩn README StegoGAN: 0.25 - 0.30)
    "lambda_consistency": 1.0,    # Trọng số nhất quán vùng đối ứng sạch (chuẩn README StegoGAN: 1.0)
    "resnet_layer": 8,            # Vị trí chèn đặc trưng ẩn & tách mặt nạ sau khối ResNet thứ 9 (0..8, chuẩn README: 8)
    "fusionblock": True,          # Bật khối ResNet hợp nhất đặc trưng ẩn trong lúc train (chuẩn README: --fusionblock)
    "mask_group": 256,            # Số kênh mặt nạ NetMatchability ở không gian tiềm ẩn (256)
    "noise_std": 0.01,            # Độ lệch chuẩn nhiễu Gauss chống giấu tin trên fake_A (chuẩn stego_gan_model.py: 0.01)
    "pool_size": 50,
    "save_every_epochs": 10,
    "probe_every_epochs": 10,     # Dùng best_fcos_checkpoint chấm nhanh AP50 trên tập con val để lưu best_net_G_A.pth (0 = tắt)
    "probe_subset_size": 128,

    # --- 4. Cấu hình Đánh giá Kép (Khớp 100% thông số với Notebook 1 để so sánh công bằng) ---
    "eval_batch_size": 4,
    "num_workers": 2,
    "fcos_fallback_img_size": 640,
    "fcos_score_thresh": 0.001,
    "pseudo_label_confs": (0.3, 0.5),
    "kid_subsets": 50,
    "kid_subset_size": 100,
    "eval_real_ir_upper_bound": True,
    "num_visual_samples": 6,

    "device": "cuda",
    "seed": 42,
}

In [ ]:
# ==============================================================================
# CELL 2: KHỞI TẠO MÔI TRƯỜNG, QUÉT INPUT KAGGLE & KIỂM TOÁN CHỐNG RÒ RỈ (UNPAIRED AUDIT)
# ==============================================================================
import contextlib
import copy
import gc
import hashlib
import io
import json
import math
import os
import random
import subprocess
import sys
import time
import xml.etree.ElementTree as ET
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
from PIL import Image
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.functional as TF

try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    HAS_PYCOCOTOOLS = True
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycocotools"])
        from pycocotools.coco import COCO
        from pycocotools.cocoeval import COCOeval
        HAS_PYCOCOTOOLS = True
    except Exception:
        HAS_PYCOCOTOOLS = False
        print("[INFO] Không cài được pycocotools (môi trường offline); sẽ dùng bộ tính Exact 101-Point COCO AP nội bộ.")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def dataloader_worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def atomic_torch_save(obj: Any, target_path: Path) -> None:
    target_path = Path(target_path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = target_path.with_suffix(target_path.suffix + ".tmp")
    torch.save(obj, tmp_path)
    os.replace(tmp_path, target_path)


def scan_kaggle_inputs_hint() -> None:
    kaggle_in = Path("/kaggle/input")
    if not kaggle_in.is_dir():
        return
    print("\n[GỢI Ý ĐƯỜNG DẪN TÌM THẤY TRONG /kaggle/input]:")
    splits = list(kaggle_in.rglob("align_train.txt"))
    pths = list(kaggle_in.rglob("*.pth"))
    jsons = list(kaggle_in.rglob("*.json"))
    for s in splits[:10]:
        print(f"  • data_root gợi ý           : \"{s.parent}\"")
    for p in pths[:20]:
        print(f"  • checkpoint (.pth) tìm thấy: \"{p}\"")
    for j in jsons[:15]:
        print(f"  • file JSON tìm thấy        : \"{j}\"")


seed_everything(CONFIG["seed"])
DEVICE = torch.device("cuda" if (CONFIG["device"] == "cuda" and torch.cuda.is_available()) else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FLIR_CLASSES: Tuple[str, ...] = ("person", "car", "bicycle")
FLIR_CLASS_TO_IDX: Dict[str, int] = {name: idx for idx, name in enumerate(FLIR_CLASSES)}
IGNORE_LABELS = {"FLIR", "dog"}
IR_EXTENSIONS = (".jpeg", ".jpg", ".png")


def resolve_split_file(root: Path, split: str) -> Path:
    filename = f"align_{split}.txt"
    for candidate in (root / filename, root / "ImageSets" / "Main" / filename):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Không tìm thấy '{filename}' trong {root}")


def read_split_stems(split_file: Path) -> List[str]:
    return [line.strip() for line in split_file.read_text(encoding="utf-8").splitlines() if line.strip()]


def ir_stem_to_rgb_path(root: Path, stem: str) -> Optional[Path]:
    base = stem.replace("_PreviewData", "")
    for ext in (".jpg", ".JPG", ".jpeg", ".png"):
        candidate = root / "JPEGImages" / f"{base}_RGB{ext}"
        if candidate.is_file():
            return candidate
    return None


def ir_stem_to_ir_path(root: Path, stem: str) -> Optional[Path]:
    for ext in IR_EXTENSIONS:
        candidate = root / "JPEGImages" / f"{stem}{ext}"
        if candidate.is_file():
            return candidate
    return None


def parse_voc_xml(xml_path: Path, min_area: float = 0.0) -> Tuple[torch.Tensor, torch.Tensor]:
    if not xml_path.is_file():
        return torch.zeros((0, 4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)
    root = ET.parse(xml_path).getroot()
    boxes, labels = [], []
    for obj in root.findall("object"):
        name = obj.findtext("name", default="").strip()
        if name in IGNORE_LABELS or name not in FLIR_CLASS_TO_IDX:
            continue
        bnd = obj.find("bndbox")
        if bnd is None:
            continue
        x1 = float(bnd.findtext("xmin"))
        y1 = float(bnd.findtext("ymin"))
        x2 = float(bnd.findtext("xmax"))
        y2 = float(bnd.findtext("ymax"))
        if x2 > x1 and y2 > y1 and (x2 - x1) * (y2 - y1) >= min_area:
            boxes.append([x1, y1, x2, y2])
            labels.append(FLIR_CLASS_TO_IDX[name])
    if not boxes:
        return torch.zeros((0, 4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)


def partition_disjoint_stems(root: Path, source_fraction: float = 0.5, split_seed: int = -1) -> Tuple[List[str], List[str], List[str]]:
    train_stems = read_split_stems(resolve_split_file(root, "train"))
    val_stems = read_split_stems(resolve_split_file(root, "validation"))
    if len(train_stems) < 2:
        raise ValueError("align_train.txt cần tối thiểu 2 mẫu để tách disjoint.")
    if len(set(train_stems)) != len(train_stems):
        raise ValueError("align_train.txt có stem trùng lặp; vi phạm điều kiện disjoint!")

    ordered = list(train_stems)
    if split_seed >= 0:
        random.Random(split_seed).shuffle(ordered)
    midpoint = int(len(ordered) // 2) if source_fraction == 0.5 else int(round(len(ordered) * source_fraction))
    rgb_source_stems = ordered[:midpoint]
    ir_target_stems = ordered[midpoint:]

    assert set(rgb_source_stems).isdisjoint(set(ir_target_stems)), "LỖI RÒ RỈ: Tập RGB nguồn và IR đích có chung stem!"
    assert set(rgb_source_stems).isdisjoint(set(val_stems)), "LỖI RÒ RỈ: Tập RGB train trùng với validation!"
    assert set(ir_target_stems).isdisjoint(set(val_stems)), "LỖI RÒ RỈ: Tập IR train trùng với validation!"
    return rgb_source_stems, ir_target_stems, val_stems


class UnpairedFLIRRGB2IRDataset(Dataset):
    """
    Dataset huấn luyện Unpaired 100% cho StegoGAN:
      - Miền A (RGB): đúng 3 kênh [3, H, W] trong [-1, 1] từ nửa đầu align_train.txt
      - Miền B (IR) : đúng 1 kênh [1, H, W] trong [-1, 1] từ nửa sau align_train.txt (bốc ngẫu nhiên độc lập!)
    """

    def __init__(
        self,
        root: Path,
        rgb_stems: Sequence[str],
        ir_stems: Sequence[str],
        *,
        load_short_edge: int = 320,
        crop_size: int = 288,
    ) -> None:
        assert set(rgb_stems).isdisjoint(set(ir_stems)), "RGB stems và IR stems bắt buộc rời nhau hoàn toàn!"
        self.rgb_paths = [p for s in rgb_stems if (p := ir_stem_to_rgb_path(root, s)) is not None]
        self.ir_paths = [p for s in ir_stems if (p := ir_stem_to_ir_path(root, s)) is not None]
        if not self.rgb_paths or not self.ir_paths:
            raise RuntimeError(f"Không tìm thấy ảnh hợp lệ: RGB={len(self.rgb_paths)}, IR={len(self.ir_paths)}")
        self.load_short_edge = int(load_short_edge)
        self.crop_size = int(crop_size)

    def __len__(self) -> int:
        return max(len(self.rgb_paths), len(self.ir_paths))

    def _resize_and_independent_crop(self, pil_img: Image.Image) -> Image.Image:
        w, h = pil_img.size
        scale = max(self.load_short_edge / min(w, h), self.crop_size / min(w, h))
        new_w = max(self.crop_size, int(round(w * scale)))
        new_h = max(self.crop_size, int(round(h * scale)))
        pil_img = pil_img.resize((new_w, new_h), Image.BICUBIC)
        top = int(torch.randint(0, new_h - self.crop_size + 1, (1,)).item())
        left = int(torch.randint(0, new_w - self.crop_size + 1, (1,)).item())
        pil_img = pil_img.crop((left, top, left + self.crop_size, top + self.crop_size))
        if float(torch.rand((), dtype=torch.float32).item()) < 0.5:
            pil_img = TF.hflip(pil_img)
        return pil_img

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        rgb_path = self.rgb_paths[idx % len(self.rgb_paths)]
        ir_idx = int(torch.randint(0, len(self.ir_paths), (1,)).item())
        ir_path = self.ir_paths[ir_idx]

        rgb_pil = self._resize_and_independent_crop(Image.open(rgb_path).convert("RGB"))
        ir_pil = self._resize_and_independent_crop(Image.open(ir_path).convert("L"))  # BẮT BUỘC 1 CHANNEL ('L')

        real_A = TF.to_tensor(rgb_pil) * 2.0 - 1.0  # [3, H, W] in [-1, 1]
        real_B = TF.to_tensor(ir_pil) * 2.0 - 1.0   # [1, H, W] in [-1, 1]
        assert real_A.shape[0] == 3 and real_B.shape[0] == 1
        return {"A": real_A, "B": real_B, "A_path": str(rgb_path), "B_path": str(ir_path)}


class FLIRValidationPairDataset(Dataset):
    def __init__(self, root: Path, stems: Sequence[str], min_area: float = 0.0) -> None:
        self.root = root
        self.records = []
        for stem in stems:
            rgb_p = ir_stem_to_rgb_path(root, stem)
            ir_p = ir_stem_to_ir_path(root, stem)
            xml_p = root / "Annotations" / f"{stem}.xml"
            if rgb_p is not None and ir_p is not None and xml_p.is_file():
                self.records.append((stem, rgb_p, ir_p, xml_p))
        if not self.records:
            raise RuntimeError(f"Không tìm thấy mẫu validation hợp lệ trong {root}")
        self.min_area = min_area

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        stem, rgb_p, ir_p, xml_p = self.records[idx]
        rgb_t = TF.to_tensor(Image.open(rgb_p).convert("RGB"))
        ir_1ch = TF.to_tensor(Image.open(ir_p).convert("L"))
        boxes, labels = parse_voc_xml(xml_p, min_area=self.min_area)
        return {
            "stem": stem,
            "rgb": rgb_t,
            "ir_1ch": ir_1ch,
            "target": {"boxes": boxes, "labels": labels, "stem": stem},
        }


def val_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "stems": tuple(item["stem"] for item in batch),
        "rgb": [item["rgb"] for item in batch],
        "ir_1ch": [item["ir_1ch"] for item in batch],
        "targets": [item["target"] for item in batch],
    }


gpu_name = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU (CHƯA BẬT GPU!)"
print(f"[ENV] Device = {DEVICE} ({gpu_name}) | PyTorch = {torch.__version__} | pycocotools = {HAS_PYCOCOTOOLS}")

DATA_ROOT = Path(CONFIG["data_root"])
if not DATA_ROOT.is_dir():
    scan_kaggle_inputs_hint()
    if CONFIG.get("strict_preflight_check", True):
        raise FileNotFoundError(
            f"[PREFLIGHT DỪNG SỚM] Không tìm thấy data_root='{DATA_ROOT}'. "
            "Hãy copy đường dẫn gợi ý bên trên vào Cell 1 (CONFIG['data_root'])!"
        )
else:
    RGB_TRAIN_STEMS, IR_TRAIN_STEMS, VAL_STEMS = partition_disjoint_stems(
        DATA_ROOT, CONFIG["source_fraction"], CONFIG["split_seed"]
    )
    train_unpaired_ds = UnpairedFLIRRGB2IRDataset(
        DATA_ROOT,
        RGB_TRAIN_STEMS,
        IR_TRAIN_STEMS,
        load_short_edge=CONFIG["load_short_edge"],
        crop_size=CONFIG["crop_size"],
    )
    val_eval_ds = FLIRValidationPairDataset(DATA_ROOT, VAL_STEMS)
    probe_sample = train_unpaired_ds[0]
    assert probe_sample["A"].shape == (3, CONFIG["crop_size"], CONFIG["crop_size"])
    assert probe_sample["B"].shape == (1, CONFIG["crop_size"], CONFIG["crop_size"])
    print("=" * 76)
    print("[KIỂM TOÁN DỮ LIỆU UNPAIRED & KÊNH THÀNH CÔNG — STEGO-GAN]")
    print(f"  • Miền A (RGB Train - 3 kênh) : {len(train_unpaired_ds.rgb_paths):4d} ảnh | Tensor shape = {tuple(probe_sample['A'].shape)}")
    print(f"  • Miền B (IR Train  - 1 kênh) : {len(train_unpaired_ds.ir_paths):4d} ảnh | Tensor shape = {tuple(probe_sample['B'].shape)}")
    print("  • Giao miền A ∩ B             : 0 ảnh (100% Unpaired / Không trùng cảnh)")
    print(f"  • Tập Validation độc lập      : {len(val_eval_ds):4d} ảnh")
    print("=" * 76)

In [ ]:
# ==============================================================================
# CELL 3: KIẾN TRÚC STEGO-GAN GỐC (CVPR 2024) + BỘ XUẤT TRỌNG SỐ KHỚP 100% CYCLEGAN REPO
# ==============================================================================
class _ResidualBlock(nn.Module):
    """Khối dư chuẩn dùng chung cho cả CycleGANGenerator và StegoGAN."""

    def __init__(self, channels: int):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3, bias=True),
            nn.InstanceNorm2d(channels), nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3, bias=True),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x: Tensor) -> Tensor:
        return x + self.conv_block(x)


class CycleGANGenerator(nn.Module):
    """
    Khớp 100% models/cyclegan_generator.py trong DAOD-Duccuong!
    Đầu vào: RGB [B, 3, H, W] trong [-1, 1] -> Đầu ra: IR [B, 1, H, W] trong [-1, 1].
    """

    def __init__(self, ngf: int = 64):
        super().__init__()
        layers: List[nn.Module] = [
            nn.ReflectionPad2d(3), nn.Conv2d(3, ngf, 7, bias=True),
            nn.InstanceNorm2d(ngf), nn.ReLU(inplace=True),
        ]
        channels = ngf
        for _ in range(2):
            layers.extend([
                nn.Conv2d(channels, channels * 2, 3, stride=2, padding=1, bias=True),
                nn.InstanceNorm2d(channels * 2), nn.ReLU(inplace=True),
            ])
            channels *= 2
        layers.extend(_ResidualBlock(channels) for _ in range(9))
        for _ in range(2):
            layers.extend([
                nn.ConvTranspose2d(channels, channels // 2, 3, stride=2, padding=1, output_padding=1, bias=True),
                nn.InstanceNorm2d(channels // 2), nn.ReLU(inplace=True),
            ])
            channels //= 2
        layers.extend([nn.ReflectionPad2d(3), nn.Conv2d(ngf, 1, 7, bias=True), nn.Tanh()])
        self.model = nn.Sequential(*layers)

    def forward(self, images: Tensor) -> Tensor:
        return self.model(images)


class NetMatchability(nn.Module):
    """
    Mô-đun học mặt nạ đối ứng của StegoGAN (sian-wusidi/StegoGAN/models/networks.py):
    Dự đoán latent_mask thuộc [0, 1] để cô lập các đặc trưng phi song ánh (chỉ có ở IR hoặc RGB).
    """

    def __init__(self, input_dim: int = 256, out_dim: int = 256):
        super().__init__()
        self.conv1 = nn.Conv2d(input_dim, input_dim, kernel_size=3, stride=1, padding=1)
        self.norm1 = nn.InstanceNorm2d(input_dim, eps=1e-5)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(input_dim, input_dim, kernel_size=3, stride=1, padding=1)
        self.norm2 = nn.InstanceNorm2d(input_dim, eps=1e-5)
        self.conv3 = nn.Conv2d(input_dim, out_dim, kernel_size=3, stride=1, padding=1)
        self.sigmoid = nn.Sigmoid()

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, feat: Tensor) -> Tensor:
        x = self.relu(self.norm1(self.conv1(feat)))
        x = self.relu(self.norm2(self.conv2(x)))
        return self.sigmoid(self.conv3(x))


def stego_mask_generate(latent_feat: Tensor, mask_net: NetMatchability) -> Tuple[Tensor, Tensor, Tensor]:
    """Tách đặc trưng khớp miền (kept) và đặc trưng phi song ánh bị loại bỏ (discarded)."""
    latent_mask = mask_net(latent_feat)
    kept = latent_feat * latent_mask
    reverse_mask = 1.0 - latent_mask  # Vùng đặc trưng phi song ánh (unmatchable)
    reverse_mask_max = torch.max(reverse_mask, dim=1, keepdim=True)[0]
    discarded = latent_feat * reverse_mask
    return kept, discarded, reverse_mask_max


class StegoResnetMaskV1Generator(nn.Module):
    """
    Mạng sinh thuận netG_A (RGB 3ch -> IR 1ch) chuẩn StegoGAN (ResnetMaskV1Generator):
      - Lúc train: nhận thêm extrafeature (đặc trưng bị loại từ miền đích) qua nhánh fusion để khép vòng chu trình mà KHÔNG ép nhánh sạch phải giấu tin!
      - Lúc inference (extrafeature=None): tạo ảnh sạch fake_B_clean [B, 1, H, W], có cấu trúc 100% tương đương CycleGANGenerator!
    """

    def __init__(
        self,
        input_nc: int = 3,
        output_nc: int = 1,
        ngf: int = 64,
        n_blocks: int = 9,
        resnet_layer: int = 8,
        fusionblock: bool = True,
    ):
        super().__init__()
        assert -1 <= resnet_layer <= n_blocks - 1
        self.resnet_layer = int(resnet_layer)
        self.n_blocks = int(n_blocks)
        self.fusion_block = bool(fusionblock)

        c1: List[nn.Module] = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, ngf, kernel_size=7, padding=0, bias=True),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True),
        ]
        for i in range(2):
            mult = 2 ** i
            c1.extend([
                nn.Conv2d(ngf * mult, ngf * mult * 2, kernel_size=3, stride=2, padding=1, bias=True),
                nn.InstanceNorm2d(ngf * mult * 2),
                nn.ReLU(inplace=True),
            ])
        self.conv1 = nn.Sequential(*c1)

        bottleneck_ch = ngf * 4
        self.conv2 = nn.Sequential(*[_ResidualBlock(bottleneck_ch) for _ in range(self.resnet_layer + 1)])
        self.conv3 = nn.Sequential(*[_ResidualBlock(bottleneck_ch) for _ in range(self.resnet_layer + 1, self.n_blocks)])

        dec: List[nn.Module] = []
        for i in range(2):
            mult = 2 ** (2 - i)
            dec.extend([
                nn.ConvTranspose2d(ngf * mult, (ngf * mult) // 2, kernel_size=3, stride=2, padding=1, output_padding=1, bias=True),
                nn.InstanceNorm2d((ngf * mult) // 2),
                nn.ReLU(inplace=True),
            ])
        dec.extend([nn.ReflectionPad2d(3), nn.Conv2d(ngf, output_nc, kernel_size=7, padding=0, bias=True), nn.Tanh()])
        self.decoder = nn.Sequential(*dec)

        if self.fusion_block:
            self.fusion = nn.Sequential(_ResidualBlock(bottleneck_ch))

    def forward(self, x: Tensor, extrafeature: Optional[Tensor] = None) -> Tensor:
        out = self.conv1(x)
        if extrafeature is not None and self.fusion_block:
            extrafeature = self.fusion(extrafeature)

        if self.resnet_layer == -1:
            if extrafeature is not None:
                out = out + extrafeature
            out = self.conv3(out)
        elif self.resnet_layer == self.n_blocks - 1:
            out = self.conv2(out)
            if extrafeature is not None:
                out = out + extrafeature
        else:
            out = self.conv2(out)
            if extrafeature is not None:
                out = out + extrafeature
            out = self.conv3(out)
        return self.decoder(out)


class StegoResnetMaskV3Generator(nn.Module):
    """
    Mạng sinh ngược netG_B (IR 1ch -> RGB 3ch) chuẩn StegoGAN (ResnetMaskV3Generator):
    Tách mặt nạ bất khả đối ứng (NetMatchability) trên đặc trưng IR 1 kênh để bảo vệ nhánh thuận khỏi ảo giác!
    """

    def __init__(
        self,
        input_nc: int = 1,
        output_nc: int = 3,
        ngf: int = 64,
        n_blocks: int = 9,
        out_dim: int = 256,
        resnet_layer: int = 8,
    ):
        super().__init__()
        self.resnet_layer = int(resnet_layer)
        self.n_blocks = int(n_blocks)

        c1: List[nn.Module] = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, ngf, kernel_size=7, padding=0, bias=True),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(inplace=True),
        ]
        for i in range(2):
            mult = 2 ** i
            c1.extend([
                nn.Conv2d(ngf * mult, ngf * mult * 2, kernel_size=3, stride=2, padding=1, bias=True),
                nn.InstanceNorm2d(ngf * mult * 2),
                nn.ReLU(inplace=True),
            ])
        self.conv1 = nn.Sequential(*c1)

        bottleneck_ch = ngf * 4
        self.conv2 = nn.Sequential(*[_ResidualBlock(bottleneck_ch) for _ in range(self.resnet_layer + 1)])
        self.conv3 = nn.Sequential(*[_ResidualBlock(bottleneck_ch) for _ in range(self.resnet_layer + 1, self.n_blocks)])
        self.mask = NetMatchability(input_dim=bottleneck_ch, out_dim=out_dim)

        dec: List[nn.Module] = []
        for i in range(2):
            mult = 2 ** (2 - i)
            dec.extend([
                nn.ConvTranspose2d(ngf * mult, (ngf * mult) // 2, kernel_size=3, stride=2, padding=1, output_padding=1, bias=True),
                nn.InstanceNorm2d((ngf * mult) // 2),
                nn.ReLU(inplace=True),
            ])
        dec.extend([nn.ReflectionPad2d(3), nn.Conv2d(ngf, output_nc, kernel_size=7, padding=0, bias=True), nn.Tanh()])
        self.decoder = nn.Sequential(*dec)

    def forward(self, x_ir_1ch: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
        out = self.conv1(x_ir_1ch)
        if self.resnet_layer == -1:
            out, discarded, rev_mask_sum = stego_mask_generate(out, self.mask)
            out = self.conv3(out)
        elif self.resnet_layer == self.n_blocks - 1:
            out = self.conv2(out)
            out, discarded, rev_mask_sum = stego_mask_generate(out, self.mask)
        else:
            out = self.conv2(out)
            out, discarded, rev_mask_sum = stego_mask_generate(out, self.mask)
            out = self.conv3(out)
        return self.decoder(out), discarded, rev_mask_sum


def export_stego_ga_to_cyclegan_state_dict(netG_A: StegoResnetMaskV1Generator) -> OrderedDict:
    """
    Ánh xạ chính xác đường suy luận sạch (extrafeature=None) của StegoGAN netG_A sang khóa phẳng
    model.0..model.26 của CycleGANGenerator (models/cyclegan_generator.py), kèm kiểm chứng sai số < 1e-6!
    """
    flat_modules = list(netG_A.conv1.children()) + list(netG_A.conv2.children()) + list(netG_A.conv3.children()) + list(netG_A.decoder.children())
    exported_gen = CycleGANGenerator(ngf=64)
    target_modules = list(exported_gen.model.children())
    assert len(flat_modules) == len(target_modules), f"Lệch số tầng: {len(flat_modules)} != {len(target_modules)}"

    with torch.no_grad():
        for src_m, dst_m in zip(flat_modules, target_modules):
            dst_m.load_state_dict(src_m.state_dict(), strict=True)

        # Kiểm tra tương đương số học tuyệt đối trên tensor ngẫu nhiên trước khi lưu!
        dev = next(netG_A.parameters()).device
        exported_gen = exported_gen.to(dev).eval()
        was_train = netG_A.training
        netG_A.eval()
        test_x = torch.randn(1, 3, 64, 64, device=dev).clamp(-1.0, 1.0)
        diff = float((netG_A(test_x, None) - exported_gen(test_x)).abs().max().item())
        if was_train:
            netG_A.train()
        assert diff < 1e-5, f"Lỗi xuất trọng số StegoGAN -> CycleGANGenerator: max_diff={diff}"

    return OrderedDict((k, v.detach().cpu().clone()) for k, v in exported_gen.state_dict().items())


class PatchGANDiscriminator(nn.Module):
    """70x70 PatchGAN Discriminator chuẩn (dùng input_nc=1 cho D_A miền IR và input_nc=3 cho D_B miền RGB)."""

    def __init__(self, input_nc: int, ndf: int = 64, n_layers: int = 3):
        super().__init__()
        kw, padw = 4, 1
        layers: List[nn.Module] = [
            nn.Conv2d(input_nc, ndf, kernel_size=kw, stride=2, padding=padw),
            nn.LeakyReLU(0.2, inplace=True),
        ]
        nf_mult = 1
        for n in range(1, n_layers):
            nf_mult_prev = nf_mult
            nf_mult = min(2 ** n, 8)
            layers.extend([
                nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kernel_size=kw, stride=2, padding=padw, bias=True),
                nn.InstanceNorm2d(ndf * nf_mult),
                nn.LeakyReLU(0.2, inplace=True),
            ])
        nf_mult_prev = nf_mult
        nf_mult = min(2 ** n_layers, 8)
        layers.extend([
            nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kernel_size=kw, stride=1, padding=padw, bias=True),
            nn.InstanceNorm2d(ndf * nf_mult),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * nf_mult, 1, kernel_size=kw, stride=1, padding=padw),
        ])
        self.model = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        return self.model(x)


def init_normal_weights_preserve_mask(net: nn.Module, gain: float = 0.02) -> None:
    """Khởi tạo Normal(0, 0.02) chuẩn StegoGAN nhưng giữ nguyên khởi tạo Kaiming riêng của NetMatchability."""
    mask_modules = set()
    for m in net.modules():
        if isinstance(m, NetMatchability):
            mask_modules.update(m.modules())

    for m in net.modules():
        if m in mask_modules:
            continue
        classname = m.__class__.__name__
        if hasattr(m, "weight") and m.weight is not None and ("Conv" in classname or "Linear" in classname):
            nn.init.normal_(m.weight.data, 0.0, gain)
            if hasattr(m, "bias") and m.bias is not None:
                nn.init.constant_(m.bias.data, 0.0)


class ImagePool:
    def __init__(self, pool_size: int = 50):
        self.pool_size = pool_size
        self.images: List[Tensor] = []

    def query(self, images: Tensor) -> Tensor:
        if self.pool_size <= 0:
            return images
        out = []
        for image in images:
            image = image.detach().unsqueeze(0)
            if len(self.images) < self.pool_size:
                self.images.append(image)
                out.append(image)
            elif random.random() > 0.5:
                idx = random.randint(0, self.pool_size - 1)
                tmp = self.images[idx].clone()
                self.images[idx] = image
                out.append(tmp)
            else:
                out.append(image)
        return torch.cat(out, dim=0)


class FrozenGANTranslator(nn.Module):
    """Khớp 100% training/gan_bridge.py."""

    def __init__(self, generator: nn.Module):
        super().__init__()
        self.generator = generator
        self.requires_grad_(False)
        self.eval()

    def train(self, mode: bool = True):
        return super().train(False)

    @torch.no_grad()
    def translate_1ch(self, images_rgb_01: Tensor) -> Tensor:
        if images_rgb_01.ndim != 4 or images_rgb_01.shape[1] != 3:
            raise ValueError(f"Input phải có dạng [B, 3, H, W], nhận được {tuple(images_rgb_01.shape)}")
        height, width = images_rgb_01.shape[-2:]
        pad_h = max(8, (height + 3) // 4 * 4) - height
        pad_w = max(8, (width + 3) // 4 * 4) - width
        outputs_1ch = []
        for image in images_rgb_01.split(1):
            gan_input = F.pad(image * 2.0 - 1.0, (0, pad_w, 0, pad_h), mode="replicate")
            raw_out = self.generator(gan_input)
            if raw_out.shape != (1, 1, height + pad_h, width + pad_w):
                raise ValueError(f"Generator bắt buộc xuất 1 kênh [1,1,H,W], nhận được {tuple(raw_out.shape)}")
            thermal_1ch = ((raw_out[:, :, :height, :width] + 1.0) * 0.5).clamp(0.0, 1.0)
            outputs_1ch.append(thermal_1ch)
        return torch.cat(outputs_1ch, dim=0)

    @torch.no_grad()
    def forward(self, images_rgb_01: Tensor) -> Tensor:
        return self.translate_1ch(images_rgb_01).repeat(1, 3, 1, 1)


def load_gan_translator(checkpoint_path: str | Path, device: torch.device) -> FrozenGANTranslator:
    path = Path(checkpoint_path)
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy GAN checkpoint: {path}")
    state = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    if all(k.startswith("module.") for k in state):
        state = {k.removeprefix("module."): v for k, v in state.items()}
    gen = CycleGANGenerator()
    gen.load_state_dict(state, strict=True)
    return FrozenGANTranslator(gen).to(device)

In [ ]:
# ==============================================================================
# CELL 4: BỘ NẠP ĐÓNG BĂNG BEST FCOS DETECTOR, PREFLIGHT CHECK & 2 BỘ ĐÁNH GIÁ (ĐỒNG BỘ 100% VỚI NOTEBOOK 1)
# ==============================================================================
from torchvision.models import Inception_V3_Weights, inception_v3
from torchvision.models.detection import fcos_resnet50_fpn
from torchvision.models.detection.fcos import FCOSHead


class _RegressionHeadWithoutCenterness(nn.Module):
    def __init__(self, conv: nn.Module, bbox_reg: nn.Module, num_anchors: int):
        super().__init__()
        self.conv = conv
        self.bbox_reg = bbox_reg
        self.num_anchors = num_anchors

    def forward(self, features: Sequence[Tensor]) -> Tensor:
        outputs = []
        for feature in features:
            reg = F.relu(self.bbox_reg(self.conv(feature)))
            b, _, h, w = reg.shape
            outputs.append(reg.view(b, self.num_anchors, 4, h, w).permute(0, 3, 4, 1, 2).reshape(b, -1, 4))
        return torch.cat(outputs, dim=1)


class _StandaloneFCOSIoUHead(nn.Module):
    def __init__(self, base_head: FCOSHead, num_classes: int = 3):
        super().__init__()
        base_cls = base_head.classification_head
        base_reg = base_head.regression_head
        in_channels = next(m.in_channels for m in base_cls.conv if isinstance(m, nn.Conv2d))
        num_anchors = base_cls.num_anchors
        cls_head = copy.deepcopy(base_cls)
        cls_head.num_classes = num_classes
        cls_head.cls_logits = nn.Conv2d(in_channels, num_anchors * num_classes, kernel_size=3, stride=1, padding=1)

        self.classification_head = cls_head
        self.regression_head = _RegressionHeadWithoutCenterness(
            conv=copy.deepcopy(base_reg.conv),
            bbox_reg=copy.deepcopy(base_reg.bbox_reg),
            num_anchors=num_anchors,
        )
        self.quality_predictor = nn.Conv2d(in_channels, num_anchors, kernel_size=3, stride=1, padding=1)
        self.num_classes = num_classes

    @staticmethod
    def _flatten(out: Tensor, c_per_anchor: int) -> Tensor:
        b, _, h, w = out.shape
        return out.view(b, -1, c_per_anchor, h, w).permute(0, 3, 4, 1, 2).reshape(b, -1, c_per_anchor)

    def forward(self, features: Sequence[Tensor]) -> Dict[str, Tensor]:
        cls_outs, qual_outs = [], []
        for f in features:
            cf = self.classification_head.conv(f)
            cls_outs.append(self._flatten(self.classification_head.cls_logits(cf), self.num_classes))
            qual_outs.append(self._flatten(self.quality_predictor(cf), 1))
        return {
            "cls_logits": torch.cat(cls_outs, dim=1),
            "bbox_regression": self.regression_head(features),
            "quality_logits": torch.cat(qual_outs, dim=1),
        }


class _StandaloneIoUFCOSInference(nn.Module):
    def __init__(self, detector: nn.Module):
        super().__init__()
        self.detector = detector

    @torch.no_grad()
    def forward(self, images: List[Tensor]) -> List[Dict[str, Tensor]]:
        orig_sizes = [(int(img.shape[-2]), int(img.shape[-1])) for img in images]
        transformed, _ = self.detector.transform(images, None)
        feats = self.detector.backbone(transformed.tensors)
        if isinstance(feats, Tensor):
            feats = OrderedDict([("0", feats)])
        feat_list = list(feats.values())
        head_out = self.detector.head(feat_list)
        anchors_list = self.detector.anchor_generator(transformed, feat_list)
        anchors_per_loc = self.detector.anchor_generator.num_anchors_per_location()
        num_anchors_per_level = [
            int(f.shape[-2] * f.shape[-1] * anchors_per_loc[lvl])
            for lvl, f in enumerate(feat_list)
        ]
        anchors_per_level = [list(a.split(num_anchors_per_level, dim=0)) for a in anchors_list]
        per_level = {k: list(v.split(num_anchors_per_level, dim=1)) for k, v in head_out.items()}
        native = self.detector.postprocess_detections(
            {
                "cls_logits": per_level["cls_logits"],
                "bbox_regression": per_level["bbox_regression"],
                "bbox_ctrness": per_level["quality_logits"],
            },
            anchors_per_level,
            list(transformed.image_sizes),
        )
        return self.detector.transform.postprocess(native, list(transformed.image_sizes), orig_sizes)


class FrozenFCOSJudge(nn.Module):
    def __init__(self, predictor: nn.Module, fmt_name: str, fixed_resize: Optional[int] = None):
        super().__init__()
        self.predictor = predictor
        self.fmt_name = fmt_name
        self.fixed_resize = fixed_resize
        self.requires_grad_(False)
        self.eval()

    def train(self, mode: bool = True):
        return super().train(False)

    @torch.no_grad()
    def predict(self, images_3ch_01: List[Tensor]) -> List[Dict[str, Tensor]]:
        if self.fixed_resize is not None:
            resized_imgs = []
            scales = []
            for img in images_3ch_01:
                h, w = img.shape[-2:]
                scales.append((w / self.fixed_resize, h / self.fixed_resize))
                resized_imgs.append(
                    F.interpolate(img.unsqueeze(0), size=(self.fixed_resize, self.fixed_resize), mode="bilinear", align_corners=False).squeeze(0)
                )
            preds = self.predictor(resized_imgs)
            restored = []
            for pred, (sx, sy) in zip(preds, scales):
                boxes = pred["boxes"].clone()
                if boxes.numel() > 0:
                    boxes[:, [0, 2]] *= sx
                    boxes[:, [1, 3]] *= sy
                restored.append({"boxes": boxes, "scores": pred["scores"], "labels": pred["labels"]})
            return restored
        return self.predictor(images_3ch_01)


def load_frozen_fcos_detector(
    checkpoint_path: str | Path,
    device: torch.device,
    *,
    score_thresh: float = 0.001,
    fallback_img_size: int = 640,
) -> FrozenFCOSJudge:
    import pickle
    import types

    path = Path(checkpoint_path)
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy best_fcos_checkpoint tại: {path}")

    class _FallbackUnpickledObj:
        """Placeholder an toàn thay cho dataclass/Enum của repo (vd: configs.types.Phase) khi chạy độc lập trên Kaggle."""
        def __new__(cls, *args, **kwargs):
            inst = super().__new__(cls)
            inst._args = args
            return inst

        def __init__(self, *args, **kwargs):
            self._args = args
            self.__dict__.update(kwargs)

        def __setstate__(self, state):
            if isinstance(state, dict):
                self.__dict__.update(state)
            else:
                self._state = state

        def get(self, key, default=None):
            return self.__dict__.get(key, default)

        @property
        def value(self):
            return self._args[0] if getattr(self, "_args", ()) else None

        def __hash__(self):
            return hash((type(self).__name__, getattr(self, "_args", ())))

        def __eq__(self, other):
            return isinstance(other, _FallbackUnpickledObj) and getattr(self, "_args", ()) == getattr(other, "_args", ())

    class _SafeUnpickler(pickle.Unpickler):
        def find_class(self, mod_name: str, name: str):
            if mod_name == "pathlib" and name in ("PosixPath", "WindowsPath"):
                return Path
            try:
                return super().find_class(mod_name, name)
            except (ModuleNotFoundError, AttributeError):
                return type(f"_Missing_{name}", (_FallbackUnpickledObj,), {"__module__": mod_name})

    safe_pickle = types.SimpleNamespace(**{k: getattr(pickle, k) for k in dir(pickle)})
    safe_pickle.Unpickler = _SafeUnpickler
    raw = torch.load(path, map_location="cpu", weights_only=False, pickle_module=safe_pickle)

    if isinstance(raw, dict) and "student" in raw and any(k.startswith("adapter.detector.") for k in raw["student"]):
        meta = raw.get("metadata", {})
        if not isinstance(meta, dict):
            meta = vars(meta) if hasattr(meta, "__dict__") else {}
        saved_m = meta.get("model")
        if saved_m is None:
            cfg_obj = meta.get("config", {})
            if not isinstance(cfg_obj, dict):
                cfg_obj = vars(cfg_obj) if hasattr(cfg_obj, "__dict__") else {}
            saved_m = cfg_obj.get("model", {})
        if not isinstance(saved_m, dict):
            saved_m = vars(saved_m) if hasattr(saved_m, "__dict__") else {}

        num_classes = int(saved_m.get("num_classes", 3))
        raw_min = saved_m.get("min_sizes", (800,))
        min_sizes = (int(raw_min),) if isinstance(raw_min, (int, float)) else tuple(int(v) for v in raw_min)
        max_size = int(saved_m.get("max_size", 1333))
        nms_thresh = float(saved_m.get("nms_thresh", 0.6))
        topk = int(saved_m.get("topk_candidates", 1000))

        base = fcos_resnet50_fpn(
            weights=None,
            weights_backbone=None,
            num_classes=91,
            min_size=(min_sizes[-1],),
            max_size=max_size,
            score_thresh=score_thresh,
            nms_thresh=nms_thresh,
            topk_candidates=topk,
            detections_per_img=100,
        )
        base.head = _StandaloneFCOSIoUHead(base.head, num_classes=num_classes)
        det_state = {
            k.removeprefix("adapter.detector."): v
            for k, v in raw["student"].items()
            if k.startswith("adapter.detector.")
        }
        base.load_state_dict(det_state, strict=True)
        return FrozenFCOSJudge(_StandaloneIoUFCOSInference(base), fmt_name="DAOD-Adapter-FCOSIoUHead", fixed_resize=None).to(device)

    state = raw
    for key in ("model", "s_model", "state_dict", "student"):
        if isinstance(state, dict) and key in state and isinstance(state[key], dict):
            state = state[key]
            break
    if all(k.startswith("module.") for k in state):
        state = {k.removeprefix("module."): v for k, v in state.items()}

    base_std = fcos_resnet50_fpn(
        weights=None,
        weights_backbone=None,
        num_classes=len(FLIR_CLASSES),
        min_size=fallback_img_size,
        max_size=fallback_img_size,
        score_thresh=score_thresh,
        detections_per_img=100,
    )
    base_std.load_state_dict(state, strict=True)
    return FrozenFCOSJudge(base_std, fmt_name=f"Torchvision-FCOS-{fallback_img_size}", fixed_resize=fallback_img_size).to(device)


def box_iou_matrix(boxes1: Tensor, boxes2: Tensor) -> Tensor:
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=torch.float32)
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp_min(0) * (boxes1[:, 3] - boxes1[:, 1]).clamp_min(0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp_min(0) * (boxes2[:, 3] - boxes2[:, 1]).clamp_min(0)
    lt = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp_min(0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    return inter / (area1[:, None] + area2[None, :] - inter).clamp_min(1e-7)


def compute_pseudo_label_quality(
    predictions: Sequence[Dict[str, Tensor]],
    targets: Sequence[Dict[str, Tensor]],
    conf_thresh: float = 0.5,
    iou_thresh: float = 0.5,
) -> Dict[str, float]:
    tp, fp, total_gt = 0, 0, 0
    matched_ious: List[float] = []
    for pred, tgt in zip(predictions, targets):
        gt_boxes = tgt["boxes"].cpu()
        gt_labels = tgt["labels"].cpu()
        total_gt += int(gt_boxes.shape[0])

        keep = pred["scores"].cpu() >= conf_thresh
        p_boxes = pred["boxes"].cpu()[keep]
        p_scores = pred["scores"].cpu()[keep]
        p_labels = pred["labels"].cpu()[keep]
        if p_boxes.shape[0] > 0:
            order = torch.argsort(p_scores, descending=True)
            p_boxes, p_labels = p_boxes[order], p_labels[order]

        gt_used = torch.zeros((gt_boxes.shape[0],), dtype=torch.bool)
        ious = box_iou_matrix(p_boxes, gt_boxes)
        for i in range(p_boxes.shape[0]):
            same_cls = (gt_labels == p_labels[i]) & (~gt_used)
            if bool(same_cls.any()):
                cand_ious = ious[i].clone()
                cand_ious[~same_cls] = -1.0
                best_iou, best_j = cand_ious.max(dim=0)
                if float(best_iou) >= iou_thresh:
                    tp += 1
                    gt_used[best_j] = True
                    matched_ious.append(float(best_iou))
                else:
                    fp += 1
            else:
                fp += 1

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, total_gt)
    f1 = 2.0 * precision * recall / max(1e-8, precision + recall)
    return {
        "conf_thresh": conf_thresh,
        "PL_Precision": float(precision),
        "PL_Recall": float(recall),
        "PL_F1": float(f1),
        "PL_MeanMatchedIoU": float(np.mean(matched_ious)) if matched_ious else 0.0,
        "PL_BoxesPerImg": float((tp + fp) / max(1, len(targets))),
        "TP": int(tp),
        "FP": int(fp),
        "GT": int(total_gt),
    }


def _fallback_coco_ap(predictions: Sequence[Dict[str, Tensor]], targets: Sequence[Dict[str, Tensor]]) -> Dict[str, float]:
    iou_thrs = np.linspace(0.5, 0.95, 10)
    recall_thrs = np.linspace(0.0, 1.0, 101)
    ap_matrix = np.full((len(iou_thrs), len(FLIR_CLASSES)), np.nan, dtype=np.float64)

    for c_idx in range(len(FLIR_CLASSES)):
        gt_by_img = {}
        total_gt = 0
        det_records = []
        for img_id, (pred, tgt) in enumerate(zip(predictions, targets)):
            g_mask = tgt["labels"].cpu() == c_idx
            g_boxes = tgt["boxes"].cpu()[g_mask]
            gt_by_img[img_id] = g_boxes
            total_gt += int(g_boxes.shape[0])

            p_mask = pred["labels"].cpu() == c_idx
            p_boxes = pred["boxes"].cpu()[p_mask]
            p_scores = pred["scores"].cpu()[p_mask]
            for b, s in zip(p_boxes, p_scores):
                det_records.append((float(s), img_id, b.unsqueeze(0)))

        if total_gt == 0:
            continue
        det_records.sort(key=lambda x: x[0], reverse=True)
        for t_idx, thr in enumerate(iou_thrs):
            used = {img_id: np.zeros(len(boxes), dtype=bool) for img_id, boxes in gt_by_img.items()}
            tps = np.zeros(len(det_records), dtype=np.float64)
            fps = np.zeros(len(det_records), dtype=np.float64)
            for d_i, (_, img_id, box_1) in enumerate(det_records):
                g_boxes = gt_by_img[img_id]
                if len(g_boxes) == 0:
                    fps[d_i] = 1.0
                    continue
                ious = box_iou_matrix(box_1, g_boxes)[0].numpy()
                ious[used[img_id]] = -1.0
                j = int(np.argmax(ious))
                if ious[j] >= thr:
                    tps[d_i] = 1.0
                    used[img_id][j] = True
                else:
                    fps[d_i] = 1.0
            tp_cum = np.cumsum(tps)
            fp_cum = np.cumsum(fps)
            rec = tp_cum / total_gt
            prec = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)
            for i in range(len(prec) - 2, -1, -1):
                prec[i] = max(prec[i], prec[i + 1])
            inds = np.searchsorted(rec, recall_thrs, side="left")
            q = np.array([prec[idx] if idx < len(prec) else 0.0 for idx in inds])
            ap_matrix[t_idx, c_idx] = float(np.mean(q))

    out = {
        "AP": float(np.nanmean(ap_matrix)),
        "AP50": float(np.nanmean(ap_matrix[0])),
        "AP75": float(np.nanmean(ap_matrix[5])),
    }
    for c_idx, name in enumerate(FLIR_CLASSES):
        out[f"AP/{name}"] = float(np.nanmean(ap_matrix[:, c_idx]))
        out[f"AP50/{name}"] = float(ap_matrix[0, c_idx])
        out[f"AP75/{name}"] = float(ap_matrix[5, c_idx])
    return out


def compute_coco_detection_metrics(
    predictions: Sequence[Dict[str, Tensor]], targets: Sequence[Dict[str, Tensor]]
) -> Dict[str, float]:
    if not HAS_PYCOCOTOOLS:
        return _fallback_coco_ap(predictions, targets)

    categories = [{"id": i + 1, "name": name} for i, name in enumerate(FLIR_CLASSES)]
    images = [{"id": i + 1} for i in range(len(targets))]
    annotations, detections = [], []
    ann_id = 1
    for img_idx, (pred, tgt) in enumerate(zip(predictions, targets), start=1):
        for box, lbl in zip(tgt["boxes"].cpu().tolist(), tgt["labels"].cpu().tolist()):
            x1, y1, x2, y2 = box
            w, h = x2 - x1, y2 - y1
            annotations.append({
                "id": ann_id, "image_id": img_idx, "category_id": int(lbl) + 1,
                "bbox": [x1, y1, w, h], "area": w * h, "iscrowd": 0,
            })
            ann_id += 1
        for box, score, lbl in zip(
            pred["boxes"].cpu().tolist(), pred["scores"].cpu().tolist(), pred["labels"].cpu().tolist()
        ):
            x1, y1, x2, y2 = box
            detections.append({
                "image_id": img_idx, "category_id": int(lbl) + 1,
                "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(score),
            })

    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO()
        coco_gt.dataset = {"info": {}, "licenses": [], "images": images, "annotations": annotations, "categories": categories}
        coco_gt.createIndex()
        if detections:
            coco_dt = coco_gt.loadRes(detections)
        else:
            coco_dt = COCO()
            coco_dt.dataset = {"images": images, "categories": categories, "annotations": []}
            coco_dt.createIndex()

        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        coco_eval.params.catIds = [1, 2, 3]
        coco_eval.params.imgIds = list(range(1, len(targets) + 1))
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()

    stat_names = ("AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR1", "AR10", "AR100")
    metrics = {name: float(coco_eval.stats[i]) for i, name in enumerate(stat_names)}
    precision = coco_eval.eval["precision"]
    area_idx = list(coco_eval.params.areaRngLbl).index("all")
    max_det_idx = len(coco_eval.params.maxDets) - 1
    iou_thrs = np.asarray(coco_eval.params.iouThrs)
    for c_idx, c_name in enumerate(FLIR_CLASSES):
        cp = precision[:, :, c_idx, area_idx, max_det_idx]
        valid = cp[cp > -1]
        metrics[f"AP/{c_name}"] = float(valid.mean()) if valid.size else float("nan")
        for thr, tag in ((0.5, "AP50"), (0.75, "AP75")):
            m_idx = np.flatnonzero(np.isclose(iou_thrs, thr))
            v = cp[m_idx[0]][cp[m_idx[0]] > -1]
            metrics[f"{tag}/{c_name}"] = float(v.mean()) if v.size else float("nan")
    return metrics


class InceptionPool3Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        try:
            net = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, transform_input=False)
        except Exception as err:
            raise RuntimeError(
                "Không tải được InceptionV3 ImageNet weights (hãy kiểm tra đã bật 'Internet: On' trên Kaggle!)."
            ) from err
        net.fc = nn.Identity()
        net.aux_logits = False
        self.net = net
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.requires_grad_(False)
        self.eval()

    @torch.no_grad()
    def forward(self, ir_1ch_01: Tensor) -> Tensor:
        assert ir_1ch_01.shape[1] == 1
        x = F.interpolate(ir_1ch_01.repeat(1, 3, 1, 1), size=(299, 299), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        return self.net(x)


def compute_fid_and_kid(
    real_feats: np.ndarray,
    fake_feats: np.ndarray,
    *,
    kid_subsets: int = 50,
    kid_subset_size: int = 100,
    seed: int = 42,
) -> Dict[str, float]:
    r64 = torch.from_numpy(np.asarray(real_feats, dtype=np.float64))
    f64 = torch.from_numpy(np.asarray(fake_feats, dtype=np.float64))

    mu1, mu2 = r64.mean(dim=0), f64.mean(dim=0)
    r_c, f_c = r64 - mu1, f64 - mu2
    sigma1 = (r_c.T @ r_c) / max(1, r64.shape[0] - 1)
    sigma2 = (f_c.T @ f_c) / max(1, f64.shape[0] - 1)

    eps_eye = torch.eye(sigma1.shape[0], dtype=torch.float64) * 1e-6
    eigvals1, eigvecs1 = torch.linalg.eigh(sigma1 + eps_eye)
    sqrt_sigma1 = (eigvecs1 * torch.sqrt(eigvals1.clamp_min(0.0))) @ eigvecs1.T
    prod_sym = sqrt_sigma1 @ (sigma2 + eps_eye) @ sqrt_sigma1
    prod_sym = 0.5 * (prod_sym + prod_sym.T)
    trace_covmean = float(torch.sqrt(torch.linalg.eigvalsh(prod_sym).clamp_min(0.0)).sum().item())
    diff = mu1 - mu2
    fid = float((diff.dot(diff) + torch.trace(sigma1) + torch.trace(sigma2) - 2.0 * trace_covmean).clamp_min(0.0).item())

    r_np = r64.numpy()
    f_np = f64.numpy()
    rng = np.random.default_rng(seed)
    m = min(kid_subset_size, len(r_np), len(f_np))
    d = float(r_np.shape[1])
    mmds = []
    for _ in range(kid_subsets):
        x = r_np[rng.choice(len(r_np), m, replace=False)]
        y = f_np[rng.choice(len(f_np), m, replace=False)]
        k_xx = (x @ x.T / d + 1.0) ** 3
        k_yy = (y @ y.T / d + 1.0) ** 3
        k_xy = (x @ y.T / d + 1.0) ** 3
        mmd = (
            (k_xx.sum() - np.trace(k_xx)) / max(1, m * (m - 1))
            + (k_yy.sum() - np.trace(k_yy)) / max(1, m * (m - 1))
            - 2.0 * k_xy.mean()
        )
        mmds.append(float(mmd))
    return {
        "FID": fid,
        "KID_x1000_mean": float(np.mean(mmds) * 1000.0),
        "KID_x1000_std": float(np.std(mmds) * 1000.0),
    }


def _sobel_magnitude(img_1ch: Tensor) -> Tensor:
    kx = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]], device=img_1ch.device).view(1, 1, 3, 3)
    ky = torch.tensor([[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]], device=img_1ch.device).view(1, 1, 3, 3)
    gx = F.conv2d(F.pad(img_1ch, (1, 1, 1, 1), mode="replicate"), kx)
    gy = F.conv2d(F.pad(img_1ch, (1, 1, 1, 1), mode="replicate"), ky)
    return torch.sqrt(gx * gx + gy * gy + 1e-8)


def _laplacian_energy(img_1ch: Tensor) -> float:
    klap = torch.tensor([[0.0, -1.0, 0.0], [-1.0, 4.0, -1.0], [0.0, -1.0, 0.0]], device=img_1ch.device).view(1, 1, 3, 3)
    resp = F.conv2d(F.pad(img_1ch, (1, 1, 1, 1), mode="replicate"), klap)
    return float((resp ** 2).mean().item())


def _ssim_1ch(x: Tensor, y: Tensor) -> float:
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    mu_x = F.avg_pool2d(x, 11, 1, 5)
    mu_y = F.avg_pool2d(y, 11, 1, 5)
    sigma_x = F.avg_pool2d(x * x, 11, 1, 5) - mu_x ** 2
    sigma_y = F.avg_pool2d(y * y, 11, 1, 5) - mu_y ** 2
    sigma_xy = F.avg_pool2d(x * y, 11, 1, 5) - mu_x * mu_y
    ssim_map = ((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) / ((mu_x ** 2 + mu_y ** 2 + c1) * (sigma_x + sigma_y + c2))
    return float(ssim_map.mean().item())


def _signed_box_ring_contrasts(img_1ch_2d: Tensor, boxes: Tensor, labels: Tensor) -> List[Tuple[int, float]]:
    H, W = img_1ch_2d.shape
    results = []
    for box, lbl in zip(boxes.tolist(), labels.tolist()):
        x1, y1, x2, y2 = box
        cx, cy = 0.5 * (x1 + x2), 0.5 * (y1 + y2)
        bw, bh = x2 - x1, y2 - y1
        if bw < 6 or bh < 6:
            continue
        ix1, iy1 = max(0, int(round(cx - 0.4 * bw))), max(0, int(round(cy - 0.4 * bh)))
        ix2, iy2 = min(W, int(round(cx + 0.4 * bw))), min(H, int(round(cy + 0.4 * bh)))
        ox1, oy1 = max(0, int(round(cx - 0.65 * bw))), max(0, int(round(cy - 0.65 * bh)))
        ox2, oy2 = min(W, int(round(cx + 0.65 * bw))), min(H, int(round(cy + 0.65 * bh)))
        if ix2 <= ix1 + 1 or iy2 <= iy1 + 1:
            continue
        core = img_1ch_2d[iy1:iy2, ix1:ix2].reshape(-1)
        outer_patch = img_1ch_2d[oy1:oy2, ox1:ox2]
        mask = torch.ones_like(outer_patch, dtype=torch.bool)
        mask[iy1 - oy1 : iy2 - oy1, ix1 - ox1 : ix2 - ox1] = False
        ring = outer_patch[mask]
        if core.numel() < 4 or ring.numel() < 8:
            continue
        med_obj = float(core.median().item())
        med_bg = float(ring.median().item())
        mad_bg = float((ring - med_bg).abs().median().item())
        c_val = (med_obj - med_bg) / (1.4826 * mad_bg + 1e-4)
        results.append((int(lbl), float(np.clip(c_val, -15.0, 15.0))))
    return results


def _wasserstein_1d(u: Sequence[float], v: Sequence[float]) -> float:
    if not u or not v:
        return float("nan")
    u_s = np.sort(np.asarray(u, dtype=np.float64))
    v_s = np.sort(np.asarray(v, dtype=np.float64))
    grid = np.linspace(0.0, 1.0, 256)
    return float(np.mean(np.abs(np.quantile(u_s, grid) - np.quantile(v_s, grid))))


@torch.no_grad()
def evaluate_generator_comprehensive(
    *,
    method_name: str,
    translator: Optional[FrozenGANTranslator],
    val_loader: DataLoader,
    device: torch.device,
    inception_extractor: Optional[InceptionPool3Extractor],
    fcos_judge: Optional[FrozenFCOSJudge],
    pseudo_confs: Sequence[float] = (0.3, 0.5),
    kid_subsets: int = 50,
    kid_subset_size: int = 100,
    is_real_ir_anchor: bool = False,
) -> Dict[str, Any]:
    real_inc_list, fake_inc_list = [], []
    edge_cos_scores, ssim_scores, psnr_scores = [], [], []
    real_hf_energies, fake_hf_energies = [], []
    real_contrasts_all, fake_contrasts_all = [], []
    fake_person_contrasts = []

    fcos_predictions: List[Dict[str, Tensor]] = []
    fcos_targets: List[Dict[str, Tensor]] = []

    t0 = time.time()
    for batch in val_loader:
        rgb_list: List[Tensor] = batch["rgb"]
        ir_1ch_list: List[Tensor] = batch["ir_1ch"]
        targets: List[Dict[str, Tensor]] = batch["targets"]

        fcos_batch_3ch: List[Tensor] = []
        for rgb_cpu, real_ir_cpu, tgt in zip(rgb_list, ir_1ch_list, targets):
            rgb = rgb_cpu.unsqueeze(0).to(device)
            real_ir = real_ir_cpu.unsqueeze(0).to(device)

            if is_real_ir_anchor:
                eval_ir_1ch = real_ir
            else:
                assert translator is not None
                eval_ir_1ch = translator.translate_1ch(rgb)
                assert eval_ir_1ch.shape[1] == 1

            if not is_real_ir_anchor and inception_extractor is not None:
                real_inc_list.append(inception_extractor(real_ir).cpu().numpy())
                fake_inc_list.append(inception_extractor(eval_ir_1ch).cpu().numpy())

                rgb_luma = 0.2989 * rgb[:, 0:1] + 0.5870 * rgb[:, 1:2] + 0.1140 * rgb[:, 2:3]
                g_rgb = _sobel_magnitude(rgb_luma).flatten(1)
                g_fake = _sobel_magnitude(eval_ir_1ch).flatten(1)
                edge_cos_scores.append(float(F.cosine_similarity(g_rgb, g_fake, dim=1).item()))

                real_hf_energies.append(_laplacian_energy(real_ir))
                fake_hf_energies.append(_laplacian_energy(eval_ir_1ch))

                mse = float(F.mse_loss(eval_ir_1ch, real_ir).item())
                psnr_scores.append(10.0 * math.log10(1.0 / max(mse, 1e-10)))
                ssim_scores.append(_ssim_1ch(eval_ir_1ch, real_ir))

                for _, c_r in _signed_box_ring_contrasts(real_ir[0, 0], tgt["boxes"], tgt["labels"]):
                    real_contrasts_all.append(c_r)
                for lbl_f, c_f in _signed_box_ring_contrasts(eval_ir_1ch[0, 0], tgt["boxes"], tgt["labels"]):
                    fake_contrasts_all.append(c_f)
                    if lbl_f == FLIR_CLASS_TO_IDX["person"]:
                        fake_person_contrasts.append(c_f)

            fcos_batch_3ch.append(eval_ir_1ch.squeeze(0).repeat(3, 1, 1))

        if fcos_judge is not None:
            preds = fcos_judge.predict(fcos_batch_3ch)
            fcos_predictions.extend({k: v.detach().cpu() for k, v in p.items()} for p in preds)
            fcos_targets.extend(
                {k: v.detach().cpu() for k, v in t.items() if isinstance(v, Tensor)}
                for t in targets
            )

    report: Dict[str, Any] = {
        "method": method_name,
        "num_eval_images": len(val_loader.dataset),
        "elapsed_sec": round(time.time() - t0, 2),
        "generation_metrics": None,
        "fcos_detection_metrics": None,
        "pseudo_label_metrics": {},
    }

    if not is_real_ir_anchor and real_inc_list:
        real_feats = np.concatenate(real_inc_list, axis=0)
        fake_feats = np.concatenate(fake_inc_list, axis=0)
        fid_kid = compute_fid_and_kid(
            real_feats, fake_feats, kid_subsets=kid_subsets, kid_subset_size=kid_subset_size
        )
        mean_real_hf = float(np.mean(real_hf_energies))
        mean_fake_hf = float(np.mean(fake_hf_energies))
        report["generation_metrics"] = {
            **fid_kid,
            "Edge_Structure_CosSim": float(np.mean(edge_cos_scores)),
            "Thermal_Contrast_W1": _wasserstein_1d(real_contrasts_all, fake_contrasts_all),
            "Person_Positive_Contrast_Ratio": (
                float(np.mean([c > 0 for c in fake_person_contrasts])) if fake_person_contrasts else float("nan")
            ),
            "HighFreq_Energy_Ratio_Fake_over_Real": mean_fake_hf / max(mean_real_hf, 1e-12),
            "Ref_Aligned_SSIM": float(np.mean(ssim_scores)),
            "Ref_Aligned_PSNR_dB": float(np.mean(psnr_scores)),
        }

    if fcos_judge is not None and fcos_predictions:
        report["fcos_detection_metrics"] = compute_coco_detection_metrics(fcos_predictions, fcos_targets)
        for conf in pseudo_confs:
            report["pseudo_label_metrics"][f"conf_{conf:.2f}"] = compute_pseudo_label_quality(
                fcos_predictions, fcos_targets, conf_thresh=conf, iou_thresh=0.5
            )
    return report


def _sha256_short(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]


def verify_all_inputs_before_training(cfg: Dict[str, Any]) -> None:
    if not cfg.get("strict_preflight_check", True):
        return

    errors = []
    if cfg.get("device") == "cuda" and Path("/kaggle").is_dir() and not torch.cuda.is_available():
        errors.append("Bạn đang chạy trên Kaggle nhưng CHƯA BẬT GPU (Settings -> Accelerator -> GPU T4 x2 / P100)!")

    fcos_ckpt = cfg.get("best_fcos_checkpoint")
    fcos_sha = None
    if not fcos_ckpt or not Path(fcos_ckpt).is_file():
        errors.append(
            f"Không tìm thấy best_fcos_checkpoint tại '{fcos_ckpt}'. "
            "Hãy điền đúng đường dẫn checkpoint FCOS tốt nhất vào Cell 1!"
        )
    else:
        fcos_sha = _sha256_short(Path(fcos_ckpt))
        test_judge = load_frozen_fcos_detector(
            fcos_ckpt, torch.device("cpu"), score_thresh=cfg["fcos_score_thresh"], fallback_img_size=cfg["fcos_fallback_img_size"]
        )
        print(f"[PREFLIGHT OK] Đã kiểm tra nạp thành công Best FCOS Detector ({test_judge.fmt_name}): {fcos_ckpt}")
        del test_judge

    has_cyc_json = bool(cfg.get("cyclegan_eval_json") and Path(cfg["cyclegan_eval_json"]).is_file())
    has_cyc_pth = bool(cfg.get("cyclegan_weights_fallback") and Path(cfg["cyclegan_weights_fallback"]).is_file())
    if has_cyc_json:
        payload = json.loads(Path(cfg["cyclegan_eval_json"]).read_text(encoding="utf-8"))
        if "cyclegan" not in payload:
            errors.append(f"File '{cfg['cyclegan_eval_json']}' không chứa khóa 'cyclegan' hợp lệ từ Notebook 1!")
        else:
            prev_fcos = (payload.get("protocol") or {}).get("fcos_judge") or {}
            if fcos_sha and prev_fcos.get("checkpoint_sha256_16") and prev_fcos["checkpoint_sha256_16"] != fcos_sha:
                print(
                    f"[CẢNH BÁO NHẸ] best_fcos_checkpoint hiện tại ({fcos_sha}) khác mã băm với file FCOS lúc xuất JSON ở Notebook 1 "
                    f"({prev_fcos['checkpoint_sha256_16']}). Hãy dùng cùng 1 checkpoint FCOS để Bảng 2 công bằng 100%!"
                )
            print(f"[PREFLIGHT OK] Đã xác nhận file kết quả CycleGAN từ Notebook 1: {cfg['cyclegan_eval_json']}")
    elif has_cyc_pth:
        _ = load_gan_translator(cfg["cyclegan_weights_fallback"], torch.device("cpu"))
        print(f"[PREFLIGHT OK] Sẽ đo CycleGAN trực tiếp từ fallback weights: {cfg['cyclegan_weights_fallback']}")
    else:
        errors.append(
            f"Chưa tìm thấy cyclegan_eval_json='{cfg.get('cyclegan_eval_json')}' từ Notebook 1 "
            "(và cũng không đặt cyclegan_weights_fallback)."
        )

    if errors:
        scan_kaggle_inputs_hint()
        joined = "\n  - ".join(errors)
        raise FileNotFoundError(
            f"[PREFLIGHT DỪNG SỚM ĐỂ BẢO VỆ QUOTA GPU CỦA BẠN]:\n  - {joined}\n"
            "(Nếu muốn bỏ qua kiểm tra này, đặt CONFIG['strict_preflight_check'] = False ở Cell 1)"
        )


if DATA_ROOT.is_dir():
    verify_all_inputs_before_training(CONFIG)

In [ ]:
# ==============================================================================
# CELL 5: HUẤN LUYỆN UNPAIRED STEGO-GAN (RGB 3 KÊNH <-> IR 1 KÊNH PHI SONG ÁNH)
# ==============================================================================
import itertools
from torch.utils.data import Subset


def build_linear_decay_scheduler(optimizer: torch.optim.Optimizer, n_epochs: int, n_epochs_decay: int):
    def lr_lambda(epoch_idx: int) -> float:
        return 1.0 - max(0, epoch_idx - n_epochs) / float(max(1, n_epochs_decay + 1))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def _load_resume_checkpoint_compat(path: str | Path, map_location: Any) -> Any:
    """Nạp resume checkpoint kể cả khi Kaggle tự giải nén file .pth thành thư mục chứa data.pkl."""
    import tempfile, zipfile
    p = Path(path)
    if p.is_file():
        return torch.load(p, map_location=map_location, weights_only=False)
    if p.is_dir():
        nested = [c for c in sorted(p.rglob("*.pth")) if c.is_file()] + [c for c in sorted(p.rglob("*.pt")) if c.is_file()]
        if nested:
            return torch.load(nested[0], map_location=map_location, weights_only=False)
        pkls = sorted(p.rglob("data.pkl"))
        if pkls:
            root = pkls[0].parent
            fd, tmp_name = tempfile.mkstemp(suffix=".pth")
            os.close(fd)
            tmp_path = Path(tmp_name)
            try:
                print(f"[AUTO-REPACK] Phát hiện resume '{p.name}' là thư mục do Kaggle giải nén -> đang ghép tạm để nạp...")
                with zipfile.ZipFile(tmp_path, "w", compression=zipfile.ZIP_STORED) as zf:
                    for f in sorted(root.rglob("*")):
                        if f.is_file():
                            zf.write(f, "archive/" + f.relative_to(root).as_posix())
                return torch.load(tmp_path, map_location=map_location, weights_only=False)
            finally:
                if tmp_path.exists():
                    tmp_path.unlink()
    raise FileNotFoundError(f"Không tìm thấy resume checkpoint hợp lệ tại: {p}")


def train_stegogan_rgb2ir(cfg: Dict[str, Any]) -> Tuple[Path, Optional[Path]]:
    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(f"Không tìm thấy thư mục FLIR tại: {DATA_ROOT}")

    train_loader = DataLoader(
        train_unpaired_ds,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=cfg["num_workers"],
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=True,
        worker_init_fn=dataloader_worker_init_fn,
    )

    # Khởi tạo 4 mạng chuẩn StegoGAN (G_A: 3->1, G_B: 1->3 có NetMatchability, D_A: 1ch IR, D_B: 3ch RGB)
    netG_A = StegoResnetMaskV1Generator(
        input_nc=3, output_nc=1, ngf=64, n_blocks=9,
        resnet_layer=cfg["resnet_layer"], fusionblock=cfg["fusionblock"],
    ).to(DEVICE)
    netG_B = StegoResnetMaskV3Generator(
        input_nc=1, output_nc=3, ngf=64, n_blocks=9,
        out_dim=cfg["mask_group"], resnet_layer=cfg["resnet_layer"],
    ).to(DEVICE)
    netD_A = PatchGANDiscriminator(input_nc=1, ndf=64, n_layers=3).to(DEVICE)  # Phân biệt miền IR 1 kênh
    netD_B = PatchGANDiscriminator(input_nc=3, ndf=64, n_layers=3).to(DEVICE)  # Phân biệt miền RGB 3 kênh

    init_normal_weights_preserve_mask(netG_A, gain=0.02)
    init_normal_weights_preserve_mask(netG_B, gain=0.02)
    init_normal_weights_preserve_mask(netD_A, gain=0.02)
    init_normal_weights_preserve_mask(netD_B, gain=0.02)

    criterion_gan = nn.MSELoss()
    criterion_l1 = nn.L1Loss()
    fake_A_pool = ImagePool(pool_size=cfg["pool_size"])
    fake_B_pool = ImagePool(pool_size=cfg["pool_size"])

    opt_G = torch.optim.Adam(
        itertools.chain(netG_A.parameters(), netG_B.parameters()),
        lr=cfg["lrG"], betas=(0.5, 0.999),
    )
    opt_D = torch.optim.Adam(
        itertools.chain(netD_A.parameters(), netD_B.parameters()),
        lr=cfg["lrD"], betas=(0.5, 0.999),
    )

    total_epochs = int(cfg["n_epochs"]) + int(cfg["n_epochs_decay"])
    sched_G = build_linear_decay_scheduler(opt_G, cfg["n_epochs"], cfg["n_epochs_decay"])
    sched_D = build_linear_decay_scheduler(opt_D, cfg["n_epochs"], cfg["n_epochs_decay"])

    start_epoch = 1
    best_probe_ap50 = -1.0
    history: List[Dict[str, float]] = []

    if cfg.get("resume_train_state") and Path(cfg["resume_train_state"]).exists():
        ckpt = _load_resume_checkpoint_compat(cfg["resume_train_state"], map_location=DEVICE)
        netG_A.load_state_dict(ckpt["netG_A"])
        netG_B.load_state_dict(ckpt["netG_B"])
        netD_A.load_state_dict(ckpt["netD_A"])
        netD_B.load_state_dict(ckpt["netD_B"])
        opt_G.load_state_dict(ckpt["opt_G"])
        opt_D.load_state_dict(ckpt["opt_D"])
        sched_G.load_state_dict(ckpt["sched_G"])
        sched_D.load_state_dict(ckpt["sched_D"])
        start_epoch = int(ckpt["epoch"]) + 1
        best_probe_ap50 = float(ckpt.get("best_probe_ap50", -1.0))
        history = list(ckpt.get("history", []))
        print(f"[RESUME] Tiếp tục huấn luyện StegoGAN từ epoch {start_epoch}/{total_epochs} (best AP50={best_probe_ap50:.4f})")

    probe_judge = None
    probe_loader = None
    if cfg["probe_every_epochs"] > 0 and cfg.get("best_fcos_checkpoint") and Path(cfg["best_fcos_checkpoint"]).is_file():
        probe_judge = load_frozen_fcos_detector(
            cfg["best_fcos_checkpoint"],
            DEVICE,
            score_thresh=cfg["fcos_score_thresh"],
            fallback_img_size=cfg["fcos_fallback_img_size"],
        )
        n_sub = min(int(cfg["probe_subset_size"]), len(val_eval_ds))
        rng_idx = np.random.default_rng(cfg["seed"]).choice(len(val_eval_ds), size=n_sub, replace=False).tolist()
        probe_loader = DataLoader(
            Subset(val_eval_ds, rng_idx),
            batch_size=cfg["eval_batch_size"],
            shuffle=False,
            collate_fn=val_collate_fn,
        )
        print(f"[PROBE] Đã nạp Frozen FCOS ({probe_judge.fmt_name}) để chấm nhanh trên {n_sub} ảnh val mỗi {cfg['probe_every_epochs']} epochs.")

    # Hai file xuất chuẩn CycleGANGenerator (cắm thẳng vào --gan-weights của repo chính!)
    latest_g_path = OUTPUT_DIR / "latest_net_G_A.pth"
    best_g_path = OUTPUT_DIR / "best_net_G_A.pth"
    # Thêm file lưu netG_B để trực quan hóa mặt nạ phi song ánh (Unmatchability Mask) ở Cell 7
    native_gb_path = OUTPUT_DIR / "stego_native_netG_B.pth"
    resume_path = OUTPUT_DIR / "stegogan_resume_latest.pth"

    lam_A = float(cfg["lambda_A"])
    lam_B = float(cfg["lambda_B"])
    lam_reg = float(cfg["lambda_reg"])
    lam_cons = float(cfg["lambda_consistency"])
    noise_std = float(cfg["noise_std"])

    print(f"[START TRAIN] StegoGAN (CVPR 2024) | Epochs: {start_epoch}..{total_epochs} | RGB(3ch) -> IR(1ch) Clean Non-Bijective")
    for epoch in range(start_epoch, total_epochs + 1):
        t_ep = time.time()
        netG_A.train()
        netG_B.train()
        netD_A.train()
        netD_B.train()

        sum_g_adv, sum_cyc, sum_stego, sum_d = 0.0, 0.0, 0.0, 0.0

        for batch in train_loader:
            real_A = batch["A"].to(DEVICE, non_blocking=True)  # [B, 3, H, W] RGB
            real_B = batch["B"].to(DEVICE, non_blocking=True)  # [B, 1, H, W] IR 1 kênh (KHÔNG GHÉP CẶP VỚI A!)

            # --- Forward hai chu trình có tách mặt nạ phi song ánh chuẩn stego_gan_model.py ---
            fake_A, latent_real_B, latent_real_B_mask = netG_B(real_B)
            noise = noise_std * torch.randn_like(fake_A)
            rec_B = netG_A(fake_A + noise, latent_real_B)
            rec_B_clean = netG_A(fake_A, None)

            fake_B_clean = netG_A(real_A, None)  # Nhánh sinh ảnh IR 1 kênh SẠCH dùng khi Inference!
            fake_B = netG_A(real_A, latent_real_B.detach())
            rec_A, latent_fake_B, latent_fake_B_mask = netG_B(fake_B)

            target_hw = real_B.shape[-2:]
            mask_real_up = F.interpolate(latent_real_B_mask, size=target_hw, mode="bilinear", align_corners=True)
            mask_fake_up = F.interpolate(latent_fake_B_mask, size=target_hw, mode="bilinear", align_corners=True)

            # --- (1) Cập nhật cặp Generator (netG_A & netG_B) ---
            netD_A.requires_grad_(False)
            netD_B.requires_grad_(False)
            opt_G.zero_grad(set_to_none=True)

            pred_fb = netD_A(fake_B)
            pred_fa = netD_B(fake_A)
            loss_G_A = criterion_gan(pred_fb, torch.ones_like(pred_fb))
            loss_G_B = criterion_gan(pred_fa, torch.ones_like(pred_fa))

            loss_cycle_A = criterion_l1(rec_A, real_A) * lam_A
            loss_cycle_B = criterion_l1(rec_B, real_B) * lam_B

            # Chính quy hóa thưa mặt nạ bất khả đối ứng + Nhất quán trên vùng đối ứng sạch (1 - mask)
            loss_reg = lam_reg * (
                torch.mean((latent_real_B_mask + 1e-10) ** 0.5)
                + torch.mean((latent_fake_B_mask + 1e-10) ** 0.5)
            )
            loss_cons_B = lam_cons * (
                criterion_l1(rec_B_clean * (1.0 - mask_real_up), real_B * (1.0 - mask_real_up))
                + criterion_l1(fake_B_clean * (1.0 - mask_fake_up), fake_B * (1.0 - mask_fake_up))
            )

            loss_G = loss_G_A + loss_G_B + loss_cycle_A + loss_cycle_B + loss_reg + loss_cons_B
            loss_G.backward()
            opt_G.step()

            # --- (2) Cập nhật cặp Discriminator (netD_A: 1ch IR, netD_B: 3ch RGB) ---
            netD_A.requires_grad_(True)
            netD_B.requires_grad_(True)
            opt_D.zero_grad(set_to_none=True)

            fake_B_q = fake_B_pool.query(fake_B)
            pred_real_B = netD_A(real_B)
            pred_fake_B = netD_A(fake_B_q.detach())
            loss_D_A = 0.5 * (
                criterion_gan(pred_real_B, torch.ones_like(pred_real_B))
                + criterion_gan(pred_fake_B, torch.zeros_like(pred_fake_B))
            )
            loss_D_A.backward()

            fake_A_q = fake_A_pool.query(fake_A)
            pred_real_A = netD_B(real_A)
            pred_fake_A = netD_B(fake_A_q.detach())
            loss_D_B = 0.5 * (
                criterion_gan(pred_real_A, torch.ones_like(pred_real_A))
                + criterion_gan(pred_fake_A, torch.zeros_like(pred_fake_A))
            )
            loss_D_B.backward()
            opt_D.step()

            sum_g_adv += float((loss_G_A + loss_G_B).item())
            sum_cyc += float((loss_cycle_A + loss_cycle_B).item())
            sum_stego += float((loss_reg + loss_cons_B).item())
            sum_d += float((loss_D_A + loss_D_B).item())

        sched_G.step()
        sched_D.step()

        n_steps = max(1, len(train_loader))
        rec = {
            "epoch": epoch,
            "lr": float(opt_G.param_groups[0]["lr"]),
            "loss_G_Adv": sum_g_adv / n_steps,
            "loss_Cycle": sum_cyc / n_steps,
            "loss_StegoMask": sum_stego / n_steps,
            "loss_D": sum_d / n_steps,
            "sec": round(time.time() - t_ep, 1),
        }

        # Xuất trực tiếp nhánh sạch fake_B_clean của netG_A về định dạng CycleGANGenerator chuẩn của repo!
        exported_state = export_stego_ga_to_cyclegan_state_dict(netG_A)
        atomic_torch_save(exported_state, latest_g_path)
        atomic_torch_save(netG_B.state_dict(), native_gb_path)
        if epoch % max(1, cfg["save_every_epochs"]) == 0 or epoch == total_epochs:
            atomic_torch_save(exported_state, OUTPUT_DIR / f"epoch_{epoch:03d}_net_G_A.pth")

        if probe_judge is not None and probe_loader is not None and (epoch % cfg["probe_every_epochs"] == 0 or epoch == total_epochs):
            probe_gen = CycleGANGenerator(ngf=64)
            probe_gen.load_state_dict(exported_state, strict=True)
            probe_res = evaluate_generator_comprehensive(
                method_name=f"StegoGAN@ep{epoch}",
                translator=FrozenGANTranslator(probe_gen).to(DEVICE),
                val_loader=probe_loader,
                device=DEVICE,
                inception_extractor=None,
                fcos_judge=probe_judge,
                pseudo_confs=(0.5,),
            )
            ap50_cur = float(probe_res["fcos_detection_metrics"]["AP50"])
            rec["probe_AP50"] = ap50_cur
            if ap50_cur > best_probe_ap50:
                best_probe_ap50 = ap50_cur
                atomic_torch_save(exported_state, best_g_path)
                print(f"  ★ [NEW BEST STEGO-GAN @ Epoch {epoch}] Frozen FCOS Probe AP50 = {best_probe_ap50:.4f} -> {best_g_path}", flush=True)

        history.append(rec)
        atomic_torch_save(
            {
                "epoch": epoch,
                "netG_A": netG_A.state_dict(),
                "netG_B": netG_B.state_dict(),
                "netD_A": netD_A.state_dict(),
                "netD_B": netD_B.state_dict(),
                "opt_G": opt_G.state_dict(),
                "opt_D": opt_D.state_dict(),
                "sched_G": sched_G.state_dict(),
                "sched_D": sched_D.state_dict(),
                "best_probe_ap50": best_probe_ap50,
                "history": history,
            },
            resume_path,
        )
        probe_str = f" | Probe AP50={rec['probe_AP50']:.4f}" if "probe_AP50" in rec else ""
        print(
            f"Epoch [{epoch:03d}/{total_epochs:03d}] lr={rec['lr']:.2e} | "
            f"G_Adv={rec['loss_G_Adv']:.4f} | Cycle={rec['loss_Cycle']:.4f} | StegoMask={rec['loss_StegoMask']:.4f} | D={rec['loss_D']:.4f}"
            f"{probe_str} ({rec['sec']}s)",
            flush=True,
        )

    (OUTPUT_DIR / "stegogan_train_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")

    del netG_A, netG_B, netD_A, netD_B, probe_judge
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    chosen_g = best_g_path if best_g_path.is_file() else latest_g_path
    return chosen_g, (native_gb_path if native_gb_path.is_file() else None)


TRAINED_STEGOGAN_WEIGHTS: Optional[Path] = None
TRAINED_STEGOGAN_NETG_B: Optional[Path] = None
if CONFIG["run_training"] and DATA_ROOT.is_dir():
    TRAINED_STEGOGAN_WEIGHTS, TRAINED_STEGOGAN_NETG_B = train_stegogan_rgb2ir(CONFIG)
    print(f"\n[HOÀN TẤT TRAIN STEGO-GAN] Trọng số sạch tương thích CycleGANGenerator tại: {TRAINED_STEGOGAN_WEIGHTS}")
elif CONFIG.get("eval_stegogan_weights"):
    TRAINED_STEGOGAN_WEIGHTS = Path(CONFIG["eval_stegogan_weights"])
    print(f"[SKIP TRAIN] Sử dụng trọng số StegoGAN có sẵn tại: {TRAINED_STEGOGAN_WEIGHTS}")

In [ ]:
# ==============================================================================
# CELL 6: NẠP BASELINE CYCLEGAN TỪ NOTEBOOK 1 & ĐÁNH GIÁ ĐỐI SÁNH STEGO-GAN (HỖ TRỢ 2 BÊN HOẶC 3 BÊN!)
# ==============================================================================
def run_stegogan_comparison(cfg: Dict[str, Any]) -> Dict[str, Any]:
    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(f"Không tìm thấy dữ liệu FLIR tại: {DATA_ROOT}")

    val_loader = DataLoader(
        val_eval_ds,
        batch_size=cfg["eval_batch_size"],
        shuffle=False,
        num_workers=cfg["num_workers"],
        collate_fn=val_collate_fn,
    )
    inc_extractor = InceptionPool3Extractor().to(DEVICE)

    fcos_judge = None
    fcos_meta = None
    if cfg.get("best_fcos_checkpoint") and Path(cfg["best_fcos_checkpoint"]).is_file():
        fcos_path = Path(cfg["best_fcos_checkpoint"])
        fcos_judge = load_frozen_fcos_detector(
            fcos_path,
            DEVICE,
            score_thresh=cfg["fcos_score_thresh"],
            fallback_img_size=cfg["fcos_fallback_img_size"],
        )
        fcos_meta = {
            "checkpoint_name": fcos_path.name,
            "checkpoint_sha256_16": _sha256_short(fcos_path),
            "detector_format": fcos_judge.fmt_name,
            "score_thresh": cfg["fcos_score_thresh"],
        }
        print(f"[JUDGE] Đã đóng băng Best FCOS Detector: {fcos_path.name} ({fcos_judge.fmt_name})")

    results: Dict[str, Any] = {"protocol": {"split": "align_validation.txt", "num_images": len(val_eval_ds), "fcos_judge": fcos_meta}}

    # --- 1. Nạp trực tiếp đánh giá CycleGAN (& Real-IR) đã lưu từ Notebook 1 ---
    cyc_json = cfg.get("cyclegan_eval_json")
    if cyc_json and Path(cyc_json).is_file():
        print(f"\n[1/3] Đã nạp trực tiếp kết quả đánh giá CycleGAN từ Notebook 1: {cyc_json}")
        cyc_payload = json.loads(Path(cyc_json).read_text(encoding="utf-8"))
        results["cyclegan"] = cyc_payload["cyclegan"]
        if cyc_payload.get("real_ir_upper_bound") is not None:
            results["real_ir_upper_bound"] = cyc_payload["real_ir_upper_bound"]
    elif cfg.get("cyclegan_weights_fallback") and Path(cfg["cyclegan_weights_fallback"]).is_file():
        fb_path = Path(cfg["cyclegan_weights_fallback"])
        print(f"\n[1/3] Chạy đo CycleGAN từ fallback weights ({fb_path.name})...")
        cyc_trans = load_gan_translator(fb_path, DEVICE)
        results["cyclegan"] = evaluate_generator_comprehensive(
            method_name="CycleGAN (Baseline)",
            translator=cyc_trans,
            val_loader=val_loader,
            device=DEVICE,
            inception_extractor=inc_extractor,
            fcos_judge=fcos_judge,
            pseudo_confs=cfg["pseudo_label_confs"],
            kid_subsets=cfg["kid_subsets"],
            kid_subset_size=cfg["kid_subset_size"],
        )
        del cyc_trans

    # Tùy chọn: Nạp thêm kết quả F-LSeSim từ Notebook 1 để in bảng 3 bên nếu người dùng cung cấp
    fls_json = cfg.get("flsesim_eval_json_optional")
    if fls_json and Path(fls_json).is_file():
        fls_payload = json.loads(Path(fls_json).read_text(encoding="utf-8"))
        if fls_payload.get("flsesim") is not None:
            results["flsesim"] = fls_payload["flsesim"]
            print(f"  + Đã nạp thêm kết quả F-LSeSim từ Notebook 1 để so sánh 3 bên!")

    if "real_ir_upper_bound" not in results and cfg.get("eval_real_ir_upper_bound") and fcos_judge is not None:
        print("\n[2/3] Đang chấm điểm Frozen FCOS trên ảnh Real-IR thật (mốc trần tham chiếu)...")
        results["real_ir_upper_bound"] = evaluate_generator_comprehensive(
            method_name="Real_IR (Validation Anchor)",
            translator=None,
            val_loader=val_loader,
            device=DEVICE,
            inception_extractor=None,
            fcos_judge=fcos_judge,
            pseudo_confs=cfg["pseudo_label_confs"],
            is_real_ir_anchor=True,
        )

    # --- 2. Đánh giá toàn diện StegoGAN vừa huấn luyện ---
    stego_path = TRAINED_STEGOGAN_WEIGHTS
    if stego_path is not None and Path(stego_path).is_file():
        print(f"\n[3/3] Đang đánh giá toàn diện StegoGAN ({Path(stego_path).name} — nạp qua load_gan_translator strict=True)...")
        stego_trans = load_gan_translator(stego_path, DEVICE)
        results["stegogan"] = evaluate_generator_comprehensive(
            method_name=f"StegoGAN ({Path(stego_path).name})",
            translator=stego_trans,
            val_loader=val_loader,
            device=DEVICE,
            inception_extractor=inc_extractor,
            fcos_judge=fcos_judge,
            pseudo_confs=cfg["pseudo_label_confs"],
            kid_subsets=cfg["kid_subsets"],
            kid_subset_size=cfg["kid_subset_size"],
        )
        del stego_trans

    summary_path = OUTPUT_DIR / "comparison_stegogan_vs_cyclegan.json"
    summary_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\n[HOÀN TẤT ĐÁNH GIÁ] Báo cáo tổng hợp đã lưu tại -> {summary_path}")
    return results


def print_comparison_report(results: Dict[str, Any]) -> None:
    cyc = results.get("cyclegan", {})
    fls = results.get("flsesim")
    stg = results.get("stegogan", {})
    real = results.get("real_ir_upper_bound", {})
    has_fls = fls is not None

    gen_specs = [
        ("FID (↓ càng thấp càng thật)", "FID", False),
        ("KID × 10^3 (↓ không chệch cỡ mẫu nhỏ)", "KID_x1000_mean", False),
        ("Edge Structure CosSim (↑ giữ biên vật thể)", "Edge_Structure_CosSim", True),
        ("Thermal Contrast W1 (↓ khớp tương phản nhiệt)", "Thermal_Contrast_W1", False),
        ("Person Positive Contrast (↑ người sáng hơn nền)", "Person_Positive_Contrast_Ratio", True),
        ("High-Freq Ratio Fake/Real (≈1.0 ít giấu tin)", "HighFreq_Energy_Ratio_Fake_over_Real", None),
        ("Ref Val SSIM (↑ tham chiếu)", "Ref_Aligned_SSIM", True),
        ("Ref Val PSNR dB (↑ tham chiếu)", "Ref_Aligned_PSNR_dB", True),
    ]

    g_cyc = cyc.get("generation_metrics") or {}
    g_fls = (fls or {}).get("generation_metrics") or {}
    g_stg = stg.get("generation_metrics") or {}

    width_1 = 108 if has_fls else 92
    print("\n" + "=" * width_1)
    print("BẢNG 1 — CÁCH EVAL 1: CÁC METRICS SINH ẢNH HỒNG NGOẠI 1 KÊNH (IMAGE GENERATION METRICS)")
    print("=" * width_1)
    if has_fls:
        print(f"{'Metric':46s} | {'CycleGAN':>12s} | {'F-LSeSim':>12s} | {'StegoGAN':>12s} | {'Δ Stego vs Cyc':>16s}")
    else:
        print(f"{'Metric':46s} | {'CycleGAN':>12s} | {'StegoGAN':>12s} | {'Δ Stego vs Cyc':>16s}")
    print("-" * width_1)

    for label, key, higher_better in gen_specs:
        vc = g_cyc.get(key, float("nan"))
        vs = g_stg.get(key, float("nan"))
        delta = vs - vc if (np.isfinite(vc) and np.isfinite(vs)) else float("nan")
        verdict = ""
        if np.isfinite(delta) and higher_better is not None:
            good = (delta > 0) if higher_better else (delta < 0)
            verdict = " ✓" if good else ""
        if has_fls:
            vf = g_fls.get(key, float("nan"))
            print(f"{label:46s} | {vc:12.4f} | {vf:12.4f} | {vs:12.4f} | {delta:+13.4f}{verdict}")
        else:
            print(f"{label:46s} | {vc:12.4f} | {vs:12.4f} | {delta:+13.4f}{verdict}")

    if any(x and x.get("fcos_detection_metrics") for x in (cyc, fls, stg, real)):
        width_2 = 114 if has_fls else 98
        print("\n" + "=" * width_2)
        print("BẢNG 2 — CÁCH EVAL 2: ĐÓNG BĂNG BEST FCOS DETECTOR CHẤM ĐIỂM DETECTION & CHẤT LƯỢNG NHÃN GIẢ")
        print("=" * width_2)
        if has_fls:
            print(f"{'Metric (↑ càng cao càng tốt)':38s} | {'Real IR (Trần)':>14s} | {'CycleGAN':>12s} | {'F-LSeSim':>12s} | {'StegoGAN':>12s} | {'Δ Stego vs Cyc':>14s}")
        else:
            print(f"{'Metric (↑ càng cao càng tốt)':38s} | {'Real IR (Trần)':>14s} | {'CycleGAN':>12s} | {'StegoGAN':>12s} | {'Δ Stego vs Cyc':>14s}")
        print("-" * width_2)

        det_keys = [
            ("COCO mAP @[0.50:0.95]", "AP"),
            ("COCO mAP @0.50 (AP50)", "AP50"),
            ("COCO mAP @0.75 (AP75)", "AP75"),
            ("AP50 / person", "AP50/person"),
            ("AP50 / car", "AP50/car"),
            ("AP50 / bicycle (vật mảnh dễ mất)", "AP50/bicycle"),
        ]
        d_real = real.get("fcos_detection_metrics") or {}
        d_cyc = cyc.get("fcos_detection_metrics") or {}
        d_fls = (fls or {}).get("fcos_detection_metrics") or {}
        d_stg = stg.get("fcos_detection_metrics") or {}

        for label, key in det_keys:
            vr = d_real.get(key, float("nan"))
            vc = d_cyc.get(key, float("nan"))
            vs = d_stg.get(key, float("nan"))
            dl = vs - vc if (np.isfinite(vc) and np.isfinite(vs)) else float("nan")
            mark = " ★" if (np.isfinite(dl) and dl > 0) else ""
            if has_fls:
                vf = d_fls.get(key, float("nan"))
                print(f"{label:38s} | {vr:14.4f} | {vc:12.4f} | {vf:12.4f} | {vs:12.4f} | {dl:+11.4f}{mark}")
            else:
                print(f"{label:38s} | {vr:14.4f} | {vc:12.4f} | {vs:12.4f} | {dl:+11.4f}{mark}")

        conf_keys = sorted((stg.get("pseudo_label_metrics") or cyc.get("pseudo_label_metrics") or {}).keys())
        for conf_key in conf_keys:
            print("-" * width_2)
            p_real = (real.get("pseudo_label_metrics") or {}).get(conf_key, {})
            p_cyc = (cyc.get("pseudo_label_metrics") or {}).get(conf_key, {})
            p_fls = ((fls or {}).get("pseudo_label_metrics") or {}).get(conf_key, {})
            p_stg = (stg.get("pseudo_label_metrics") or {}).get(conf_key, {})
            for pl_label, pl_k in (
                (f"Nhãn giả Precision ({conf_key})", "PL_Precision"),
                (f"Nhãn giả Recall    ({conf_key})", "PL_Recall"),
                (f"Nhãn giả F1-Score  ({conf_key})", "PL_F1"),
                (f"Nhãn giả Mean IoU  ({conf_key})", "PL_MeanMatchedIoU"),
            ):
                vr = p_real.get(pl_k, float("nan"))
                vc = p_cyc.get(pl_k, float("nan"))
                vs = p_stg.get(pl_k, float("nan"))
                dl = vs - vc if (np.isfinite(vc) and np.isfinite(vs)) else float("nan")
                mark = " ★" if (np.isfinite(dl) and dl > 0) else ""
                if has_fls:
                    vf = p_fls.get(pl_k, float("nan"))
                    print(f"{pl_label:38s} | {vr:14.4f} | {vc:12.4f} | {vf:12.4f} | {vs:12.4f} | {dl:+11.4f}{mark}")
                else:
                    print(f"{pl_label:38s} | {vr:14.4f} | {vc:12.4f} | {vs:12.4f} | {dl:+11.4f}{mark}")
        print("=" * width_2)


EVAL_RESULTS: Optional[Dict[str, Any]] = None
if DATA_ROOT.is_dir():
    EVAL_RESULTS = run_stegogan_comparison(CONFIG)
    print_comparison_report(EVAL_RESULTS)

In [ ]:
# ==============================================================================
# CELL 7: TRỰC QUAN HÓA ẢNH IR 1 KÊNH STEGO-GAN, BẢN ĐỒ MẶT NẠ PHI SONG ÁNH & HỘP DỰ ĐOÁN FCOS
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def _draw_boxes(ax, boxes: Tensor, labels: Tensor, scores: Optional[Tensor] = None, conf_min: float = 0.4):
    palette = {0: "#00ff66", 1: "#ffcc00", 2: "#00e5ff"}  # person=xanh lá, car=vàng, bicycle=xanh ngọc
    for idx in range(boxes.shape[0]):
        if scores is not None and float(scores[idx]) < conf_min:
            continue
        x1, y1, x2, y2 = boxes[idx].tolist()
        cls_id = int(labels[idx])
        color = palette.get(cls_id, "red")
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.4, edgecolor=color, facecolor="none"))
        cls_name = FLIR_CLASSES[cls_id] if 0 <= cls_id < len(FLIR_CLASSES) else str(cls_id)
        caption = f"{cls_name}" if scores is None else f"{cls_name}:{float(scores[idx]):.2f}"
        ax.text(x1, max(10, y1 - 3), caption, color="black", fontsize=7, bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none"))


if DATA_ROOT.is_dir():
    stg_trans = load_gan_translator(TRAINED_STEGOGAN_WEIGHTS, DEVICE) if (TRAINED_STEGOGAN_WEIGHTS and Path(TRAINED_STEGOGAN_WEIGHTS).is_file()) else None
    viz_judge = (
        load_frozen_fcos_detector(
            CONFIG["best_fcos_checkpoint"], DEVICE, score_thresh=0.3, fallback_img_size=CONFIG["fcos_fallback_img_size"]
        )
        if (CONFIG.get("best_fcos_checkpoint") and Path(CONFIG["best_fcos_checkpoint"]).is_file())
        else None
    )

    # Nếu có netG_B vừa train, nạp để hiển thị luôn bản đồ nhiệt Mặt nạ Phi song ánh (Unmatchability Mask)
    viz_netG_B = None
    if TRAINED_STEGOGAN_NETG_B and Path(TRAINED_STEGOGAN_NETG_B).is_file():
        viz_netG_B = StegoResnetMaskV3Generator(
            input_nc=1, output_nc=3, ngf=64, n_blocks=9,
            out_dim=CONFIG["mask_group"], resnet_layer=CONFIG["resnet_layer"],
        ).to(DEVICE)
        viz_netG_B.load_state_dict(torch.load(TRAINED_STEGOGAN_NETG_B, map_location="cpu", weights_only=True))
        viz_netG_B.eval()

    chosen_indices = []
    for i in range(len(val_eval_ds)):
        lbls = set(val_eval_ds[i]["target"]["labels"].tolist())
        if FLIR_CLASS_TO_IDX["bicycle"] in lbls or FLIR_CLASS_TO_IDX["person"] in lbls:
            chosen_indices.append(i)
        if len(chosen_indices) >= CONFIG["num_visual_samples"]:
            break

    if chosen_indices:
        n_rows = len(chosen_indices)
        fig, axes = plt.subplots(n_rows, 4, figsize=(22, 4.4 * n_rows), squeeze=False)
        for r, ds_idx in enumerate(chosen_indices):
            item = val_eval_ds[ds_idx]
            rgb = item["rgb"].to(DEVICE)
            real_ir = item["ir_1ch"].to(DEVICE)
            tgt = item["target"]

            axes[r, 0].imshow(rgb.cpu().permute(1, 2, 0).numpy().clip(0, 1))
            _draw_boxes(axes[r, 0], tgt["boxes"], tgt["labels"])
            axes[r, 0].set_title(f"1. RGB + GT Boxes\n{item['stem']}", fontsize=9)

            axes[r, 1].imshow(real_ir[0].cpu().numpy(), cmap="gray", vmin=0, vmax=1)
            if viz_judge is not None:
                p_real = viz_judge.predict([real_ir.repeat(3, 1, 1)])[0]
                _draw_boxes(axes[r, 1], p_real["boxes"].cpu(), p_real["labels"].cpu(), p_real["scores"].cpu(), conf_min=0.4)
            axes[r, 1].set_title("2. Real IR (1-ch) + Frozen FCOS", fontsize=9)

            if stg_trans is not None:
                stg_1ch = stg_trans.translate_1ch(rgb.unsqueeze(0))[0]
                axes[r, 2].imshow(stg_1ch[0].cpu().numpy(), cmap="gray", vmin=0, vmax=1)
                if viz_judge is not None:
                    p_stg = viz_judge.predict([stg_1ch.repeat(3, 1, 1)])[0]
                    _draw_boxes(axes[r, 2], p_stg["boxes"].cpu(), p_stg["labels"].cpu(), p_stg["scores"].cpu(), conf_min=0.4)
            axes[r, 2].set_title("3. StegoGAN Clean IR (1-ch) + Frozen FCOS", fontsize=9)

            if viz_netG_B is not None:
                with torch.no_grad():
                    _, _, rev_mask = viz_netG_B((real_ir.unsqueeze(0) * 2.0 - 1.0))
                    mask_up = F.interpolate(rev_mask, size=real_ir.shape[-2:], mode="bilinear", align_corners=True)[0, 0]
                axes[r, 3].imshow(mask_up.cpu().numpy(), cmap="inferno", vmin=0.0, vmax=1.0)
                axes[r, 3].set_title("4. StegoGAN Non-Bijective Mask (1 - M)\n(Vùng dị thường nhiệt được cô lập)", fontsize=9)
            else:
                axes[r, 3].text(0.5, 0.5, "Mask khả dụng sau khi train", ha="center", va="center")

            for c in range(4):
                axes[r, c].axis("off")

        plt.tight_layout()
        viz_out = OUTPUT_DIR / "stegogan_fcos_probe_and_mask.png"
        fig.savefig(viz_out, dpi=150, bbox_inches="tight")
        print(f"Đã lưu ảnh đối chiếu trực quan & mặt nạ phi song ánh -> {viz_out}")
        plt.show()